In [1]:
!pip install tensorly --break-system-packages

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import hashlib
import numpy as np
import math
import time
import matplotlib.pyplot as plt
import secrets  
from datetime import datetime
from skimage import io
from PIL import Image
import os
from tensorly.decomposition import tucker
import torch.nn as nn
import tensorly as tl
tl.set_backend('numpy')

In [3]:
# Function to display the encrypted image
def display_image(image, title=""):
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')
    plt.show()

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class CNN16x16x16(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.1),  # Changed from ReLU to LeakyReLU
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.1),  # Changed from ReLU to LeakyReLU
            nn.Conv2d(64, 16, kernel_size=1),
            nn.LeakyReLU(0.1)   # Changed from ReLU to LeakyReLU
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16*16*16, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Data transforms
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

# ImageFolder datasets
train_dir = 'ham10000/classified_images_train'
val_dir = 'ham10000/classified_images_val'

print("Train dir exists:", os.path.exists(train_dir))
print("Val dir exists:", os.path.exists(val_dir))

print("Train contents:", os.listdir(train_dir))
print("Val contents:", os.listdir(val_dir))

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Model, loss, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN16x16x16(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_loss:.4f}")

    # Validation
    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            val_loss += criterion(outputs, labels).item() * imgs.size(0)
            predicted = torch.argmax(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    val_loss /= len(val_loader.dataset)
    accuracy = correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}], Val Loss: {val_loss:.4f}, Val Acc: {accuracy:.4f}")

# Save the trained feature extractor
torch.save(model.features.state_dict(), "cnn_feature_extractor_leaky.pth")
print("Saved feature extractor weights to cnn_feature_extractor_leaky.pth")


Train dir exists: True
Val dir exists: True
Train contents: ['bkl', 'nv', 'mel', 'df', 'vasc', 'bcc', 'akiec']
Val contents: ['bkl', 'nv', 'mel', 'df', 'vasc', 'bcc', 'akiec']
Epoch [1/10], Train Loss: 0.9882
Epoch [1/10], Val Loss: 0.9110, Val Acc: 0.6783
Epoch [2/10], Train Loss: 0.8587
Epoch [2/10], Val Loss: 0.9110, Val Acc: 0.6833
Epoch [3/10], Train Loss: 0.8181
Epoch [3/10], Val Loss: 0.8493, Val Acc: 0.6873
Epoch [4/10], Train Loss: 0.7636
Epoch [4/10], Val Loss: 0.8004, Val Acc: 0.7032
Epoch [5/10], Train Loss: 0.7314
Epoch [5/10], Val Loss: 0.7959, Val Acc: 0.7087
Epoch [6/10], Train Loss: 0.7046
Epoch [6/10], Val Loss: 0.7803, Val Acc: 0.7107
Epoch [7/10], Train Loss: 0.6867
Epoch [7/10], Val Loss: 0.7681, Val Acc: 0.7112
Epoch [8/10], Train Loss: 0.6623
Epoch [8/10], Val Loss: 0.7704, Val Acc: 0.7172
Epoch [9/10], Train Loss: 0.6452
Epoch [9/10], Val Loss: 0.7663, Val Acc: 0.7267
Epoch [10/10], Train Loss: 0.6316
Epoch [10/10], Val Loss: 0.7517, Val Acc: 0.7182
Saved featur

In [1]:
import numpy as np
import hashlib
from datetime import datetime
import secrets
from scipy.linalg import svd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

class RobustTensorDecomposition:
    """
    Implementation of Robust Tensor Decomposition (RTD) for image key generation
    Based on Algorithm 1 from the provided paper
    """
    
    def __init__(self, convergence_criterion=1e-6, thresholding_scale=0.01, max_iterations=100):
        self.delta = convergence_criterion
        self.beta = thresholding_scale
        self.max_iterations = max_iterations
    
    def hard_threshold(self, T, threshold):
        """
        Hard thresholding operation: H_ζ(T)
        """
        T_thresh = T.copy()
        T_thresh[np.abs(T_thresh) < threshold] = 0
        return T_thresh

    def rank_l_approximation(self, T, rank_l):
        """
        Compute rank-l approximation using gradient ascent method following Procedure 1
        from the RTD algorithm theory
        """
        # Ensure tensor is in float64 format
        T = T.astype(np.float64)
        n1, n2, n3 = T.shape
    
        # Parameters for gradient ascent
        N1 = 5  # Number of initializations/restarts
        N2 = 20  # Number of power iterations for each initialization
        max_grad_iterations = 100
        convergence_tol = 1e-6
    
        # Store the eigenpairs
        eigenpairs = []
        T_deflated = T.copy()
        n1, n2, n3 = T_deflated.shape  # n1=16, n2=16, n3=3 
    
        # Find top rank_l eigenpairs
        for j in range(rank_l):
            best_lambda = -np.inf
            best_u = None
        
            # Multiple random initializations
            for i in range(N1):
                # Random initialization
                theta = np.random.normal(0, 1, (n1, n2))
                theta = theta / np.linalg.norm(theta, 'fro')
            
                # Compute top singular vector of T_j(I, I, theta)
                T_theta = np.zeros((n1, n2))
                for k in range(n3):
                    for l in range(n2):
                        T_theta += T_deflated[:, :, k] * theta[k, l]
            
                # Get top singular vector
                U, s, Vt = np.linalg.svd(T_theta)
                u = U[:, 0] if len(U) > 0 else np.random.normal(0, 1, n)
                u = u / np.linalg.norm(u)
            
                # Power method iterations
                v = u.copy()
                for t in range(N2):
                    T_v_v = np.zeros(n1)
                    for a in range(T.shape[0]):
                        for b in range(T.shape[1]):
                            for c in range(T.shape[2]):
                                T_v_v[a] += T_deflated[a, b, c] * v[b] * v[c]
                
                    norm_T_v_v = np.linalg.norm(T_v_v)
                    if norm_T_v_v > 1e-12:
                        v = T_v_v / norm_T_v_v
                
                    lambda_val = 0
                    for a in range(T.shape[0]):
                        for b in range(T.shape[1]):
                            for c in range(T.shape[2]):
                                lambda_val += T_deflated[a, b, c] * v[a] * v[b] * v[c]
            
                if lambda_val > best_lambda:
                    best_lambda = lambda_val
                    best_u = v.copy()
        
            # Gradient ascent refinement
            if best_u is not None:
                v = best_u.copy()
                lambda_val = best_lambda
            
                for t in range(max_grad_iterations):
                    v_old = v.copy()
                
                    gradient = np.zeros(n1)
                    for a in range(T.shape[0]):
                        for b in range(T.shape[1]):
                            for c in range(T.shape[2]):
                                gradient[a] += T_deflated[a, b, c] * v[b] * v[c]
                
                    lambda_val = 0
                    for a in range(T.shape[0]):
                        for b in range(T.shape[1]):
                            for c in range(T.shape[2]):
                                lambda_val += T_deflated[a, b, c] * v[a] * v[b] * v[c]
                
                    step_size = 1.0 / (4 * abs(lambda_val) * (1 + abs(lambda_val) / np.sqrt(n1)) + 1e-8)
                    gradient_term = gradient - lambda_val * v
                    v = v + step_size * gradient_term
                
                    v_norm = np.linalg.norm(v)
                    if v_norm > 1e-12:
                        v = v / v_norm
                
                    if np.linalg.norm(v - v_old) < convergence_tol:
                        break
            
                eigenpairs.append((abs(lambda_val), v))
            
                # Deflation
                for a in range(T.shape[0]):
                    for b in range(T.shape[1]):
                        for c in range(T.shape[2]):
                            T_deflated[a, b, c] -= lambda_val * v[a] * v[b] * v[c]
    
        # Construct rank-l approximation
        T_approx = np.zeros_like(T)
        for lambda_val, u in eigenpairs:
            for a in range(T.shape[0]):
                for b in range(T.shape[1]):
                    for c in range(T.shape[2]):
                        T_approx[a, b, c] += lambda_val * u[a] * u[b] * u[c]
    
        return T_approx

    def estimate_eigenvalue(self, T, k):
        """
        Estimate the k largest eigenvalues using the gradient ascent method
        """
        T = T.astype(np.float64)
        n1, n2, n3 = T.shape
    
        N1 = 3
        N2 = 10
    
        eigenvalues = []
        T_deflated = T.copy()
    
        for j in range(min(k, n1)):
            best_lambda = -np.inf
        
            for i in range(N1):
                v = np.random.normal(0, 1, n1)
                v = v / np.linalg.norm(v)
            
                for t in range(N2):
                    T_v_v = np.zeros(n1)
                    for a in range(T.shape[0]):
                        for b in range(T.shape[1]):
                            for c in range(T.shape[2]):
                                T_v_v[a] += T_deflated[a, b, c] * v[b] * v[c]
                
                    norm_T_v_v = np.linalg.norm(T_v_v)
                    if norm_T_v_v > 1e-12:
                        v = T_v_v / norm_T_v_v
                
                    lambda_val = 0
                    for a in range(T.shape[0]):
                        for b in range(T.shape[1]):
                            for c in range(T.shape[2]):
                                lambda_val += T_deflated[a, b, c] * v[a] * v[b] * v[c]
            
                if abs(lambda_val) > best_lambda:
                    best_lambda = abs(lambda_val)
                    best_v = v.copy()
        
            if best_lambda > 1e-12:
                eigenvalues.append(best_lambda)
            
                lambda_val = 0
                for a in range(T.shape[0]):
                    for b in range(T.shape[1]):
                        for c in range(T.shape[2]):
                            lambda_val += T_deflated[a, b, c] * best_v[a] * best_v[b] * best_v[c]
            
                for a in range(T.shape[0]):
                    for b in range(T.shape[1]):
                        for c in range(T.shape[2]):
                            T_deflated[a, b, c] -= lambda_val * best_v[a] * best_v[b] * best_v[c]
            else:
                break
    
        return sorted(eigenvalues, reverse=True)
    
    def rtd_decomposition(self, T, target_rank):
        """
        Main RTD algorithm implementation
        """
        n1, n2, n3 = T.shape
        
        eigenvals = self.estimate_eigenvalue(T, target_rank + 1)
        zeta_0 = self.beta * eigenvals[0] if eigenvals else self.beta
        
        L = np.zeros_like(T)
        S = self.hard_threshold(T - L, zeta_0)
        
        for stage_l in range(1, target_rank + 1):
            norm_T_minus_S = np.linalg.norm(T - S)
            tau = max(1, int(5 * np.log(max(n1, n2, n3)) * self.beta * norm_T_minus_S**2 / self.delta))
            tau = min(tau, self.max_iterations)
            
            for t in range(tau):
                L_new = self.rank_l_approximation(T - S, stage_l)
                residual = T - L_new
                
                if t > 0:
                    eigenvals_residual = self.estimate_eigenvalue(residual, min(stage_l + 1, target_rank))
                    if len(eigenvals_residual) > stage_l:
                        sigma_l_plus_1 = eigenvals_residual[stage_l]
                        sigma_l = eigenvals_residual[stage_l - 1] if stage_l > 0 else eigenvals_residual[0]
                        zeta = self.beta * (sigma_l_plus_1 + 0.5 * t * sigma_l)
                    else:
                        zeta = zeta_0 / (t + 1)
                else:
                    zeta = zeta_0
                
                S_new = self.hard_threshold(residual, zeta)
                
                if np.linalg.norm(L_new - L) < self.delta and np.linalg.norm(S_new - S) < self.delta:
                    break
                
                L = L_new
                S = S_new
            
            eigenvals_L = self.estimate_eigenvalue(L, stage_l + 1)
            if len(eigenvals_L) > stage_l and self.beta * eigenvals_L[stage_l] < self.delta / (2 * max(n1, n2, n3)):
                break
        
        return L, S

def image_to_tensor_cnn(image):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    weights_path = 'cnn_feature_extractor.pth'  # Path to your saved weights
    extractor = CNNFeatureExtractor(device=device, weights_path=weights_path)
    tensor = extractor.extract_features(image)  # (16, 16, 16)
    return tensor

class CNNFeatureExtractor:
    """CNN-based 16x16x16 tensor generator using CNN for 64x64 RGB input."""
    def __init__(self, device='cpu', weights_path=None):
        self.device = device
        # Define only the feature extraction part
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(64, 16, kernel_size=1),
            nn.ReLU(True)
        ).to(device)
        
        if weights_path is not None:
            self.model.load_state_dict(torch.load(weights_path, map_location=device))
            print(f"[INFO] Loaded weights from {weights_path}")
            # After loading CNN weights
    tensor_raw = image_to_tensor_cnn(original_image)
    print(f"Raw tensor stats:")
    print(f"  Min: {tensor_raw.min()}")
    print(f"  Max: {tensor_raw.max()}")
    print(f"  Mean: {tensor_raw.mean()}")
    print(f"  Std: {tensor_raw.std()}")
    print(f"  Non-zeros: {np.count_nonzero(tensor_raw)} / {tensor_raw.size}")

    self.model.eval()

    self.transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor(),
        ])

    def extract_features(self, image):
        if isinstance(image, np.ndarray):
            if image.max() <= 1.0:
                image = (image * 255).astype(np.uint8)
            image = Image.fromarray(image)
        img_tensor = self.transform(image).unsqueeze(0).to(self.device)
        with torch.no_grad():
            features = self.model(img_tensor)
        features_np = features.squeeze(0).cpu().numpy().transpose(1, 2, 0)
        #NORMALIZATION:
        # fmin, fmax = features_np.min(), features_np.max()
        # if fmax - fmin > 1e-8:
        #     features_np = (features_np - fmin) / (fmax - fmin)
        return features_np






def extract_tensor_features(T, L, S):
    """
    Extract meaningful features from the original tensor (T) and its 
    low-rank (L) and sparse (S) components
    """
    features = {}
    
    # Features from low-rank component L
    features['L_frobenius_norm'] = np.linalg.norm(L.flatten())
    L_reshaped = L.reshape(L.shape[0], -1)
    features['L_spectral_norm'] = np.linalg.norm(L_reshaped, 2)
    features['L_nuclear_norm'] = np.sum(np.linalg.svd(L_reshaped, compute_uv=False))
    features['L_mean'] = np.mean(L)
    features['L_std'] = np.std(L)
    
    # Features from sparse component S
    features['S_l0_norm'] = np.count_nonzero(S)
    features['S_l1_norm'] = np.sum(np.abs(S))
    features['S_max'] = np.max(np.abs(S))
    features['S_mean'] = np.mean(S)
    
    # Combined features
    reconstruction = L + S
    features['reconstruction_error'] = np.linalg.norm((T - reconstruction).flatten()) / np.linalg.norm(T.flatten())
    features['decomposition_ratio'] = features['L_frobenius_norm'] / (features['S_l1_norm'] + 1e-10)
    
    return features




def enhanced_key_generation(image, timestamp, mu, nonce=None, target_rank=1):
    """
    Enhanced key generation using Robust Tensor Decomposition with CNN-based tensor
    
    Args:
        image: input image array
        timestamp: timestamp string
        mu: chaotic map parameter
        nonce: cryptographic nonce
        target_rank: RTD target rank
    """
    if nonce is None:
        nonce = secrets.token_hex(16)
    
    # Convert image to tensor using CNN or traditional method
    print(f"\n=== Using CNN-based tensor generation ===")
    tensor = image_to_tensor_cnn(image)  # Always returns (16, 16, 3)

    print(f"Final tensor shape: {tensor.shape}")
    
    # Apply RTD (unchanged mathematical logic)
    rtd = RobustTensorDecomposition(
        thresholding_scale=0.01,
        convergence_criterion=1e-6,
        max_iterations=100
    )
    L, S = rtd.rtd_decomposition(tensor, target_rank)
    
    # Extract channel-wise sums from the low-rank tensor L
    n1, n2, n3 = L.shape
    channel_sums = []

    for k in range(n3):
        channel_sum = np.sum(L[:, :, k])
        channel_sums.append(channel_sum)
        print(f"Channel {k} sum: {channel_sum}")

    # Convert channel sums to scaled integers
    channel_sums_scaled = []
    for i, ch_sum in enumerate(channel_sums):
        scaled_sum = int(ch_sum * 1e15) % (10**15)
        channel_sums_scaled.append(scaled_sum)

    # Create sum string
    sum_string = ''.join(str(ch_sum) for ch_sum in channel_sums_scaled) + timestamp + nonce
    print(f"Channel-wise concatenated string length: {len(sum_string)}")
    
    # Hash using SHA512
    hashed_sum = hashlib.sha512(sum_string.encode()).hexdigest()
    
    # Divide into 8 parts
    parts = [hashed_sum[i*16:(i+1)*16] for i in range(8)]
    
    keys = []
    for i, part in enumerate(parts):
        decimal_value = int(part, 16)
        first_15_digits = str(decimal_value)[:15]
        extracted_value = int(first_15_digits) if first_15_digits else 0
        normalized_value = extracted_value / 10**15
        keys.append(normalized_value)
        print(f"RTD Key {i+1}: {normalized_value}")
    
    return keys, channel_sums, L, S, tensor


def rtd_key_generation(fake_image, timestamp, mu, nonce=None, target_rank=1):
    """
    Drop-in replacement for your original key_generation function
    Now uses CNN-based 3D tensor generation
    
    Args:
        fake_image: input image
        timestamp: timestamp string
        mu: chaotic map parameter
        nonce: optional nonce
        target_rank: RTD rank
    """
    print(f"\nUsing RTD-based key generation with tensor formation...")
    
    # Generate RTD-based keys
    keys, channel_sums, L, S, tensor = enhanced_key_generation(
        fake_image, timestamp, mu, nonce, target_rank)
    
    # Extract feature vector stats
    features = extract_tensor_features(tensor, L, S)
    
    # Print RTD information
    print(f"\nRTD Decomposition Stats:")
    print(f"  - Low-rank component shape: {L.shape}")
    print(f"  - Sparse component non-zeros: {np.count_nonzero(S)}")
    print(f"  - L Frobenius norm: {features['L_frobenius_norm']:.6f}")
    print(f"  - L Spectral norm: {features['L_spectral_norm']:.6f}")
    print(f"  - L Nuclear norm: {features['L_nuclear_norm']:.6f}")
    print(f"  - S L0 norm (sparsity): {features['S_l0_norm']}")
    print(f"  - S L1 norm: {features['S_l1_norm']:.6f}")
    print(f"  - Reconstruction error: {features['reconstruction_error']:.6f}")
    print(f"  - Decomposition ratio: {features['decomposition_ratio']:.6f}")
    
    return keys


NameError: name 'original_image' is not defined

### Key Generation from DCGAN generated Images

In [ ]:
# Define the 1D Exponential Chebyshev Map (1-DEC)
def one_dec_map(y, mu):
    # Ensure y stays within [-1, 1] for valid arccos computation
    y = np.clip(y, -1, 1)
    return 1 - 2 * (np.cos(np.arccos(y) * np.exp(abs(mu)) * np.arccos(y)))**2

# Function to generate the nonce
def generate_nonce():
    return secrets.token_hex(16)  # Generate a 16-byte (128-bit) random hex string

# Function to generate chaotic sequence using the 1-DEC map
def chaotic_system(initial_value, mu, iterations=150):
    chaotic_sequence = []
    y = initial_value
    for _ in range(iterations):
        y = one_dec_map(y, mu)
        chaotic_sequence.append(y)
    return chaotic_sequence

# Function to divide hashed sum into 8 parts
def divide_into_eight_parts(hashed_sum):
    # Ensure the hashed_sum is divided into 8 equal parts (128 hex characters / 8 = 16)
    parts = [hashed_sum[i*16:(i+1)*16] for i in range(8)]
    return parts

# Function to convert part to decimal, extract first 15 digits, and normalize
def extract_and_normalize(part):
    # Step 1: Convert hex part to decimal
    decimal_value = int(part, 16)  # Convert the hexadecimal part to decimal    
    # Step 2: Extract the first 15 digits by converting to string and slicing
    first_15_digits = str(decimal_value)[:15]  # Extract the first 15 digits
    
    # Step 3: Convert the first 15 digits back to an integer
    extracted_value = int(first_15_digits)
    
    # Step 4: Normalize by dividing by 10^15
    normalized_value = extracted_value / 10**15
    return normalized_value

In [ ]:
# Main function to generate keys
# def key_generation(fake_image, timestamp, mu, nonce):
#     if nonce is None:
#         nonce = generate_nonce()  # Generate a nonce if not provided
    
#     # Separate image into RGB channels and calculate pixel sums
#     red_sum = np.sum(fake_image[:, :, 0])
#     green_sum = np.sum(fake_image[:, :, 1])
#     blue_sum = np.sum(fake_image[:, :, 2])
    
#     # Create sum string with RGB values incremented by 1, timestamp, and nonce
#     sum_string = f"{int(red_sum) + 1}{int(green_sum) + 1}{int(blue_sum) + 1}" + timestamp + nonce
#     # Hash the sum string using SHA512
#     hashed_sum = hashlib.sha512(sum_string.encode()).hexdigest()
#     # Divide the hashed sum into 8 parts for the chaotic map key generation
#     divided_parts = divide_into_eight_parts(hashed_sum)
#     keySequence = []
#     keys = []
#     i=1
#     for part in divided_parts:
#         # Convert hex part to decimal and normalize
#         normalized_part = extract_and_normalize(part)
#         print(f"key {i}:{normalized_part}")
#         keys.append(normalized_part)
#         i = i+1
#         # Generate chaotic sequence using normalized value and chaotic map (1-DEC)
#         chaotic_sequence = chaotic_system(normalized_part, mu)
#         keySequence.append(chaotic_sequence)
#     return keys, keySequence

In [ ]:
# Select 4 decoy images from DCGAN generated images
image_paths = [
    'mri.jpg',
    'AFVAE-CDL-image.png', 
    'chest-x-ray.jpg',
    'baboon.jpg'
]

# Load and resize all 4 images to 64x64
images = []
for i, path in enumerate(image_paths):
    img = Image.open(path)
    print(f"Original image {i+1} size: {img.size}")
    
    # Resize to 64x64 using high-quality resampling
    img_resized = img.resize((64, 64), Image.Resampling.LANCZOS)
    img_array = np.array(img_resized)
    images.append(img_array)
    print(f"Resized image {i+1} shape: {img_array.shape}")

# ADDED: Convert all images to RGB (3 channels)
for i in range(len(images)):
    if images[i].shape[2] == 4:  # RGBA image
        images[i] = images[i][:, :, :3]  # Remove alpha channel
        print(f"Converted image {i+1} from RGBA to RGB")
    elif images[i].shape[2] != 3:
        if images[i].shape[2] == 1:  # Grayscale
            images[i] = np.repeat(images[i], 3, axis=2)
        else:
            images[i] = images[i][:, :, :3]  # Take first 3 channels
        print(f"Converted image {i+1} to 3 channels")

# Verify all images now have 3 channels
for i, img in enumerate(images):
    print(f"Final image {i+1} shape: {img.shape}")

# Create 2x2 grid (each image is 64x64, so final grid will be 128x128)
print("\nCreating 2x2 grid...")

# Top row: concatenate images[0] and images[1] horizontally
top_row = np.concatenate([images[0], images[1]], axis=1)
print(f"Top row shape: {top_row.shape}")

# Bottom row: concatenate images[2] and images[3] horizontally  
bottom_row = np.concatenate([images[2], images[3]], axis=1)
print(f"Bottom row shape: {bottom_row.shape}")

# Final grid: concatenate top and bottom rows vertically
original_image = np.concatenate([top_row, bottom_row], axis=0)
print(f"Final 2x2 grid shape: {original_image.shape}")


# Current timestamp in the specified format
timestamp = datetime.now().strftime("%d%m%Y%H%M%S")
    
# Chaotic map parameter (mu)
mu = 2
    
# Generate a nonce
nonce = generate_nonce()
    
# Generate 8 keys in a single call
keys_original = rtd_key_generation(
    original_image, 
    timestamp, 
    mu, 
    nonce,
    target_rank=1
)


# Unpack the keys
K1, K2, K3, K4, K5, K6, K7, K8 = keys_original

# Generate chaotic sequences for each key
keys_seq = [np.array(chaotic_system(k, 2, 1500000)) for k in keys_original]

In [ ]:
# # Select a decoy image from DCGAN generated images
# image_path = 'AFVAE-CDL-image.png'
# original_image = Image.open(image_path)
# original_image = original_image.resize((64, 64))  # Resize to match encryption process
# original_image = np.array(original_image)



# # Current timestamp in the specified format
# timestamp = datetime.now().strftime("%d%m%Y%H%M%S")
    
# # Chaotic map parameter (mu)
# mu = 2
    
# # Generate a nonce
# nonce = generate_nonce()
    
# # Generate 8 keys in a single call
# keys_original = rtd_key_generation(original_image, timestamp, mu, nonce)

# # Example usage:
# # Define keys (You need to adjust the key values according to your needs)
# # K1, K2, K3, K4, K5, K6, K7, K8 = [0.123456789, 0.987654321, 0.555555555, 0.444444444, 0.333333333, 0.222222222, 0.111111111, 0.888888888]
# K1, K2, K3, K4, K5, K6, K7, K8 = keys_original

# keys_seq= [np.array(chaotic_system(k, 2, 1500000)) for k in keys_original]
# # keys_seq[3] = K4

### Encryption Algorithm

In [ ]:
# Convert the image to red, green, and blue channels; convert channel pixels to bitstream and concatenate
def convert_to_bitstream(image):
    bitstream = np.concatenate([
        np.unpackbits(image[:, :, channel], axis=None) for channel in range(3)
    ])
    return bitstream

# Create sequence S1 from the bitstream, taking 3 bits at a time
def create_sequence(bitstream):
    S1 = bitstream[:len(bitstream) // 3 * 3].reshape(-1, 3)
    return S1

# Create new sequence S2 using XOR between S1 and the previous S2 term
def create_new_sequence_using_xor(S1):
    S2 = np.empty_like(S1)  # Preallocate the array for S2
    S2[0] = S1[0]           # First term remains the same
    np.bitwise_xor.accumulate(S1, out=S2)  # Perform cumulative XOR in bulk
    return S2

# Convert S2 back to RGB image
def convert_to_rgb_image_encrypt(S2, original_shape):
    """
    Convert S2 back to an RGB image while ensuring the correct bitstream size and structure.
    """
    bitstream = S2.flatten().astype(np.uint8)
    num_pixels = original_shape[0] * original_shape[1]

    # Slice the bitstream for each channel
    red_bits, green_bits, blue_bits = np.split(bitstream[:num_pixels * 24], 3)

    # Pack bits into bytes and reshape to original channel dimensions
    red_channel = np.packbits(red_bits).reshape(original_shape[:2])
    green_channel = np.packbits(green_bits).reshape(original_shape[:2])
    blue_channel = np.packbits(blue_bits).reshape(original_shape[:2])

    # Stack channels into the final image
    return np.stack((red_channel, green_channel, blue_channel), axis=-1)


# Intershuffling step
def intershuffling(I, K1, K2, K3):
    # Get the shape of the input image
    s = I.shape
    K1 = K1[:s[0]]
    K2 = K2[:s[1]]
    K3 = K3[:s[2]]
    K1 = abs(K1) * 10**15
    K2 = abs(K2) * 10**15
    K3 = abs(K3) * 10**15
    K1 = np.array(list(map(int, K1)))
    K2 = np.array(list(map(int, K2)))
    K3 = np.array(list(map(int, K3)))
    # print(K1,K2,K3)
    # print(K2[s[1]-1])
    # Create a copy of the image to avoid modifying the original during shuffling
    shuffled_image = np.copy(I)

    # Iterate over the dimensions of the image and apply shuffling
    for i in range(s[0]):
        for j in range(s[1]):
            for k in range(s[2]):
                # Calculate new indices based on K1, K2, K3
                # print((K3[k]%3)%3)
                new_i = (K1[i]%256)% s[0]
                new_j = (K2[j]%256) % s[1] 
                new_k = (K3[k]%3) % s[2]
                # print(new_i, new_j, new_k)
                # Swap the pixel at (i, j, k) with the calculated new position
                temp = shuffled_image[i, j, k]
                shuffled_image[i, j, k] = shuffled_image[new_i, new_j, new_k]
                shuffled_image[new_i, new_j, new_k] = temp
                # print("one:", i,j, k, "two:", new_i, new_j, new_k)
    return shuffled_image

# def zigzag_xor(image, K4):
#     # # K4 = abs(K4) * 10**15
#     # # K4 = np.array(list(map(int, K4)))
#     # # K4 = K4%256
#     # K4 = int(abs(K4) * 10**15) % 256
#     # result_image = np.copy(image)
#     # for i in range(image.shape[0]):
#     #     for j in range(image.shape[1]):
#     #         for k in range(image.shape[2]):
#     #             result_image[i, j, k] = np.bitwise_xor(image[i, j, k], K4)
#     # return result_image
#     # Compute K4 as an integer modulo 256
#     K4 = int(abs(K4) * 10**15) % 256

#     # Perform XOR operation across the entire RGB image
#     result_image = np.bitwise_xor(image, K4)
#     return result_image

def zigzag_xor(image, K4):
    s = image.shape
    K4 = abs(K4) * 10**15
    K4 = np.array(list(map(int, K4)))
    K4 = K4%256
    # # K4 = int(abs(K4) * 10**15) % 256
    # result_image = np.copy(image)
    # for i in range(image.shape[0]):
    #     for j in range(image.shape[1]):
    #         for k in range(image.shape[2]):
    #             result_image[i, j, k] = np.bitwise_xor(image[i, j, k], K4[i*s[1]*s[2]+j*s[2]+k])
    # return result_image

    # # Calculate chaotic sequence K4 values modulo 256
    # s = image.shape
    # K4 = (np.abs(K4) * 10**15).astype(np.uint64) % 256
    
    # # Flatten K4 and ensure it matches the total number of image elements
    # K4 = np.resize(K4, image.size)  # Reshape or tile K4 to match the image size

    # # Flatten the image, apply XOR, and reshape back to the original shape
    # result_image = np.bitwise_xor(image.flatten(), K4).reshape(s)

    # return result_image

     # Calculate chaotic sequence K4 values modulo 256
    # K4 = (np.abs(K4) * 10**15).astype(np.int64) % 256

    # Flatten K4 and match it to the total number of image elements
    K4 = np.resize(K4, image.size).astype(np.uint8)

    # Create a copy of the original image
    result_image = np.copy(image)

    # Flatten the image for efficient processing
    flat_image = result_image.flatten()

    # Perform XOR operation
    flat_result = np.bitwise_xor(flat_image, K4)

    # Reshape the result back to the original image shape
    result_image = flat_result.reshape(s).astype(np.uint8)

    return result_image


def planet(I, K1):
    # Get the size of the input image
    ImageSize = I.shape
    # Extract the red, green, and blue channels
    RedChannel = I[:, :, 0]
    GreenChannel = I[:, :, 1]
    BlueChannel = I[:, :, 2]
    
    # Reshape the channels into 1D bitstreams
    Redbitstream = RedChannel.flatten()
    Greenbitstream = GreenChannel.flatten()
    Bluebitstream = BlueChannel.flatten()

    # Concatenate the bitstreams from all channels
    TotalBits = np.concatenate((Redbitstream, Greenbitstream, Bluebitstream))
    
    # Initialize an empty array for the mixed bitstream
    Mixedbitstream = np.zeros_like(Redbitstream)

    # Initialize an empty array for the mixed bitstream
    Mixedbitstream = np.zeros_like(TotalBits)

    # K1 = np.array(chaotic_system(K1, 2, len(TotalBits)))
    
    # Scale K1 by 256 and take the absolute value
    K1 = abs(K1) * 256
    # Iterate through the image size
    for i in range(len(TotalBits)):
        if i % 3 == 0:
            # XOR with the red bitstream
            Mixedbitstream[i] = np.bitwise_xor(Redbitstream[i // 3], np.uint8(K1[i]))
        elif i % 3 == 1:
            # XOR with the green bitstream
            Mixedbitstream[i] = np.bitwise_xor(Greenbitstream[i // 3], np.uint8(K1[i]))
        else:
            # XOR with the blue bitstream
            Mixedbitstream[i] = np.bitwise_xor(Bluebitstream[i // 3], np.uint8(K1[i]))
    
    # Reshape the mixed bitstream back into the image size
    RestoredImage = Mixedbitstream.reshape(ImageSize[0], ImageSize[1], ImageSize[2])
    
    return RestoredImage



# Complete encryption process
def encrypt_image(image, key):
    # Step 2: Perform VPD

    K1, K2, K3, K4, K5, K6, K7, K8 = key
    start = time.time()
    
    bitstream = convert_to_bitstream(image)
    S1 = create_sequence(bitstream)
    S2 = create_new_sequence_using_xor(S1)
    I1 = convert_to_rgb_image_encrypt(S2, image.shape)
    end = time.time()
    print(f"Encryption Time1: {end-start:.6f} seconds")
    # display_image(I1, "I1")
    # Step 3: Intershuffling using K1, K2, K3
    start = time.time()
    I2 = intershuffling(I1, K1, K2, K3)
    end = time.time()
    print(f"Encryption Time2: {end-start:.6f} seconds")
    # display_image(I2, "I2")
    # Step 4: Zigzag XORing using K4
    start = time.time()
    I3 = zigzag_xor(I2, K4)
    end = time.time()
    print(f"Encryption Time3: {end-start:.6f} seconds")
    # display_image(I3, "I3")
    # Step 5: Perform VPD again
    start = time.time()
    bitstream = convert_to_bitstream(I3)
    S1 = create_sequence(bitstream)
    S2 = create_new_sequence_using_xor(S1)
    I4 = convert_to_rgb_image_encrypt(S2, image.shape)
    end = time.time()
    print(f"Encryption Time4: {end-start:.6f} seconds")
    start = time.time()
    
    # display_image(I4, "I4")
    # Step 6: Intershuffling using K5, K6, K7
    I5 = intershuffling(I4, K5, K6, K7)
    end = time.time()
    print(f"Encryption Time5: {end-start:.6f} seconds")
    start = time.time()
    # display_image(I5, "I5")
    # Step 7: Zigzag XORing using K4 again
    I6 = zigzag_xor(I5, K4)
    end = time.time()
    print(f"Encryption Time6: {end-start:.6f} seconds")
    start = time.time()

    # display_image(I6, "I6")
    # Step 8: Final VPD
    bitstream = convert_to_bitstream(I6)
    bits = bitstream
    S1 = create_sequence(bitstream)
    # print("I6 S1:", S1)
    S2 = create_new_sequence_using_xor(S1)
    # print("ENCRYPT S2 BITSTREAM:", len(S2))
    # print("ENCRYPT S2 BITSTREAM:", S2)
    I7 = convert_to_rgb_image_encrypt(S2, image.shape)
    end = time.time()
    print(f"Encryption Time7: {end-start:.6f} seconds")
    start = time.time()
    # display_image(I7, title="I7")
    # Step 9: Planet encryption using K8
    encrypted_image = planet(I7, K8)
    end = time.time()
    print(f"Encryption Time8: {end-start:.6f} seconds")
    
    return encrypted_image

### Decryption Algorithm

In [ ]:
#TODO: Optimize the functions here used in decryption to reduce decryption time

def PlanetDec(C, K8):
    # Get image size and reshape the encrypted image into a flat bitstream
    image_size = C.shape
    mixed_bitstream = C.flatten()
    K8=K8[:len(mixed_bitstream)]
    # K8 = np.array(chaotic_system(K8, 2, len(mixed_bitstream)))
    K8 = abs(K8) * 256
    K8 = K8.astype(np.uint8)

    # Initialize bitstreams for red, green, and blue channels
    red_bitstream = np.zeros(len(mixed_bitstream) // 3, dtype=np.uint8)
    green_bitstream = np.zeros(len(mixed_bitstream) // 3, dtype=np.uint8)
    blue_bitstream = np.zeros(len(mixed_bitstream) // 3, dtype=np.uint8)
    # Reverse the XOR operation with K8
    for i in range(len(mixed_bitstream)):
        if i % 3 == 0:
            red_bitstream[i // 3] = np.bitwise_xor(mixed_bitstream[i], K8[i]) #CHANGE THIS LATER
        elif i % 3 == 1:
            green_bitstream[i // 3] = np.bitwise_xor(mixed_bitstream[i], K8[i])
        else:
            blue_bitstream[i // 3] = np.bitwise_xor(mixed_bitstream[i], K8[i])
    # Reshape the bitstreams into their original color channel shapes
    red_channel = red_bitstream.reshape(image_size[0], image_size[1])
    green_channel = green_bitstream.reshape(image_size[0], image_size[1])
    blue_channel = blue_bitstream.reshape(image_size[0], image_size[1])

    # Combine the color channels into an RGB image
    restored_image = np.zeros(image_size, dtype=np.uint8)
    restored_image[:, :, 0] = red_channel
    restored_image[:, :, 1] = green_channel
    restored_image[:, :, 2] = blue_channel

    return restored_image

# Step 2: Convert image to bitstream
def ConvertToBitstream(I):
    image = np.copy(I)
    red_channel = np.unpackbits(image[:, :, 0], axis=1)
    green_channel = np.unpackbits(image[:, :, 1], axis=1)
    blue_channel = np.unpackbits(image[:, :, 2], axis=1)
    bitstream = np.concatenate((red_channel.flatten(), green_channel.flatten(), blue_channel.flatten()))
    return bitstream

# Step 3: Create sequence S1
def CreateSequence(bitstream):
    S1 = []
    for i in range(0, len(bitstream), 3):
        S1.append(bitstream[i:i+3])  # Group bits into sequences of 3
    return np.array(S1)

# Step 4: Reverse the XOR sequence (generate S2)
def ReverseSequenceUsingXOR(S1):
    S2 = np.zeros_like(S1)
    S2[0] = S1[0]  # Start with the first element of S1 (this might need to match the encryption more closely)
    for i in range(1, len(S1)):
        S2[i] = np.bitwise_xor(S1[i], S1[i - 1])  # Reverse XOR operation, ensure this mirrors encryption
    return np.array(S2)

def ConvertToRGBImage(S2, image_shape):
    bitstream = np.concatenate(S2).astype(np.uint8)
    
    # Calculate the number of bits per channel (8 bits per pixel)
    num_pixels = image_shape[0] * image_shape[1]
    
    red_bits = bitstream[:num_pixels * 8]
    green_bits = bitstream[num_pixels * 8:2 * num_pixels * 8]
    blue_bits = bitstream[2 * num_pixels * 8:]
    red_channel = np.packbits(red_bits).reshape((image_shape[0], image_shape[1]))
    green_channel = np.packbits(green_bits).reshape((image_shape[0], image_shape[1]))
    blue_channel = np.packbits(blue_bits).reshape((image_shape[0], image_shape[1]))
    # Return the stacked image without the extra dimension
    return np.stack((red_channel, green_channel, blue_channel), axis=-1)


# Step 6: Reverse Zigzag XOR
def ReverseZigzagXOR(image, K4):

    s = image.shape
    K4 = abs(K4) * 10**15
    K4 = np.array(list(map(int, K4)))
    K4 = K4%256
    # # K4 = int(abs(K4) * 10**15) % 256
    # result_image = np.copy(image)
    # for i in range(image.shape[0]):
    #     for j in range(image.shape[1]):
    #         for k in range(image.shape[2]):
    #             result_image[i, j, k] = np.bitwise_xor(image[i, j, k], K4[i*s[1]*s[2]+j*s[2]+k])
    # return result_image

    # Flatten K4 and match it to the total number of image elements
    K4 = np.resize(K4, image.size).astype(np.uint8)

    # Create a copy of the original image
    result_image = np.copy(image)

    # Flatten the image for efficient processing
    flat_image = result_image.flatten()

    # Perform XOR operation
    flat_result = np.bitwise_xor(flat_image, K4)

    # Reshape the result back to the original image shape
    result_image = flat_result.reshape(s).astype(np.uint8)

    return result_image


# Step 7: Reverse Intershuffling
def ReverseInterShuffling(I, K1, K2, K3):
    # Get the shape of the input image
    s = I.shape
    # K1 = np.array(chaotic_system(K1, 2, s[0]))
    # K2 = np.array(chaotic_system(K2, 2, s[1]))
    # K3 = np.array(chaotic_system(K3, 2, s[2]))
    # print(k1,k2,k3)
    K1 = K1[:s[0]]
    K2 = K2[:s[1]]
    K3 = K3[:s[2]]
    K1 = abs(K1) * 10**15
    K2 = abs(K2) * 10**15
    K3 = abs(K3) * 10**15
    K1 = np.array(list(map(int, K1)))
    K2 = np.array(list(map(int, K2)))
    K3 = np.array(list(map(int, K3)))
    
    
    # print(K2[s[1]-1])
    # Create a copy of the image to avoid modifying the original during reverse shuffling
    reverse_shuffled_image = np.copy(I)
    
    # Iterate over the dimensions of the image in reverse order to reverse the shuffling process
    for i in range(s[0]-1, -1, -1):
        for j in range(s[1]-1, -1, -1):
            for k in range(s[2]-1, -1, -1):
                orig_i = (K1[i] % 256) % s[0]
                orig_j = (K2[j] % 256) % s[1]
                orig_k = (K3[k] % 3) % s[2]
                
                # Swap back to the original position
                temp = reverse_shuffled_image[i, j, k]
                reverse_shuffled_image[i, j, k] = reverse_shuffled_image[orig_i, orig_j, orig_k]
                reverse_shuffled_image[orig_i, orig_j, orig_k] = temp
                # print("one:", i,j, k, "two:",  orig_i, orig_j, orig_k)
                

    return reverse_shuffled_image

# Step 8: Decrypt Image (Main Function)
def decrypt_image(encrypted_image, key):
    K1, K2, K3, K4, K5, K6, K7, K8 = key
    image_shape = encrypted_image.shape[:2]  # Extract the original image dimensions
    # Step 1: Reverse planet encryption with K8
    I7 = PlanetDec(encrypted_image, K8)

    # display_image(I7, "Decrypted I7")
    # Step 2: Reverse VPD
    bitstream = ConvertToBitstream(I7)   
    S1 = CreateSequence(bitstream)
    S2 = ReverseSequenceUsingXOR(S1)
    I6 = ConvertToRGBImage(S2, image_shape)

    # display_image(I6, "Decrypted I6")
    # Step 3: Reverse Zigzag XORing using K4
    I5 = ReverseZigzagXOR(I6, K4)
    # display_image(I5, "decrypted I5")
    # Step 4: Reverse Intershuffling with K5, K6, K7
    I4 = ReverseInterShuffling(I5, K5, K6, K7)
    # display_image(I4, "decrypted I4")
    # Step 5: Reverse VPD again
    bitstream = ConvertToBitstream(I4)
    S1 = CreateSequence(bitstream)
    S2 = ReverseSequenceUsingXOR(S1)
    I3 = ConvertToRGBImage(S2, image_shape)
    # display_image(I3, "decrypted I3")
    # Step 6: Reverse Zigzag XORing using K4 again
    I2 = ReverseZigzagXOR(I3, K4)
    # display_image(I2, "decrypted I2")
    # Step 7: Reverse Intershuffling with K1, K2, K3
    I1 = ReverseInterShuffling(I2, K1, K2, K3)
    # display_image(I1, "decrypted I1")
    # Step 8: Reverse VPD for the final time
    bitstream = ConvertToBitstream(I1)
    S1 = CreateSequence(bitstream)
    S2 = ReverseSequenceUsingXOR(S1)
    Decrypted_Image = ConvertToRGBImage(S2, image_shape)
    end_time = time.time()
    return Decrypted_Image

In [ ]:
# Now encrypt the 2x2 grid image using the generated keys
encrypted_image = encrypt_image(original_image, keys_seq)

# Display the original 2x2 grid image
display_image(original_image, "Original 2x2 Grid Image")

# Display the encrypted image
display_image(encrypted_image, "Encrypted Image")

# Decrypt the image using the same keys
decrypted_image = decrypt_image(encrypted_image, keys_seq)

# Display the decrypted image
display_image(decrypted_image, "Decrypted Image")

In [ ]:
# Load the image to encrypt
image_path = 'images/617_3,10.jpeg'
original_image = Image.open(image_path)
original_image = np.array(original_image)


# Encrypt the image using the generated keys
encrypted_image = encrypt_image(original_image, keys_seq)

display_image(original_image, "Original Image")
# Display the encrypted image
display_image(encrypted_image, "Encrypted Image")

# Decrypt the image using the keys
decrypted_image = decrypt_image(encrypted_image,  keys_seq)

display_image(decrypted_image, "decrypted Image")


In [ ]:
display_image(original_image);
display_image(encrypted_image);
display_image(decrypted_image);

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import hashlib
import secrets
from datetime import datetime
from scipy.linalg import sqrtm
import time

def comprehensive_histogram_analysis(original_image, encrypted_image, decrypted_image, save_path=None):
    """
    Generate comprehensive histogram analysis for encryption algorithm evaluation
    Format: (a-c) red channel, (d-f) green channel, (g-i) blue channel
    Each row shows: original, encrypted, decrypted
    """
    
    # Create figure with 3x3 subplots
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    fig.suptitle('Histogram Analysis of Encryption Algorithm\n(Original, Encrypted, Decrypted)', 
                 fontsize=16, fontweight='bold')
    
    # Define colors for each channel
    colors = ['red', 'green', 'blue']
    channel_names = ['Red Channel', 'Green Channel', 'Blue Channel']
    
    # Labels for columns
    col_labels = ['Original', 'Encrypted', 'Decrypted']
    
    # Process each channel (Red, Green, Blue)
    for channel in range(3):
        # Extract channel data
        orig_channel = original_image[:, :, channel].flatten()
        enc_channel = encrypted_image[:, :, channel].flatten()
        dec_channel = decrypted_image[:, :, channel].flatten()
        
        # Plot histograms for each state (original, encrypted, decrypted)
        images_data = [orig_channel, enc_channel, dec_channel]
        
        for col in range(3):
            ax = axes[channel, col]
            
            # Calculate histogram
            hist, bins = np.histogram(images_data[col], bins=256, range=(0, 256))
            
            # Plot histogram
            ax.hist(images_data[col], bins=256, range=(0, 256), 
                   color=colors[channel], alpha=0.7, edgecolor='black', linewidth=0.5)
            
            # Customize plot
            ax.set_xlim(0, 255)
            ax.set_xlabel('Pixel Intensity')
            ax.set_ylabel('Frequency')
            ax.grid(True, alpha=0.3)
            
            # Add subplot labels
            if channel == 0:  # First row
                ax.set_title(f'({chr(97 + col)}) {col_labels[col]}', fontweight='bold')
            elif channel == 1:  # Second row  
                ax.set_title(f'({chr(100 + col)}) {col_labels[col]}', fontweight='bold')
            else:  # Third row
                ax.set_title(f'({chr(103 + col)}) {col_labels[col]}', fontweight='bold')
            
            # Add channel label on the left
            if col == 0:
                ax.text(-0.15, 0.5, channel_names[channel], rotation=90, 
                       transform=ax.transAxes, ha='center', va='center', 
                       fontweight='bold', fontsize=12)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Histogram analysis saved to: {save_path}")
    
    plt.show()
    
    return fig

def calculate_histogram_statistics(original_image, encrypted_image, decrypted_image):
    """
    Calculate statistical metrics for histogram analysis
    """
    stats = {}
    
    for channel in range(3):
        channel_name = ['Red', 'Green', 'Blue'][channel]
        
        # Extract channel data
        orig_channel = original_image[:, :, channel].flatten()
        enc_channel = encrypted_image[:, :, channel].flatten()
        dec_channel = decrypted_image[:, :, channel].flatten()
        
        # Calculate statistics
        stats[channel_name] = {
            'Original': {
                'mean': np.mean(orig_channel),
                'std': np.std(orig_channel),
                'min': np.min(orig_channel),
                'max': np.max(orig_channel),
                'entropy': calculate_entropy(orig_channel)
            },
            'Encrypted': {
                'mean': np.mean(enc_channel),
                'std': np.std(enc_channel),
                'min': np.min(enc_channel),
                'max': np.max(enc_channel),
                'entropy': calculate_entropy(enc_channel)
            },
            'Decrypted': {
                'mean': np.mean(dec_channel),
                'std': np.std(dec_channel),
                'min': np.min(dec_channel),
                'max': np.max(dec_channel),
                'entropy': calculate_entropy(dec_channel)
            }
        }
    
    return stats

def calculate_entropy(data):
    """Calculate entropy of image data"""
    hist, _ = np.histogram(data, bins=256, range=(0, 256))
    hist = hist[hist > 0]  # Remove zero bins
    prob = hist / np.sum(hist)
    entropy = -np.sum(prob * np.log2(prob))
    return entropy

def print_histogram_statistics(stats):
    """Print formatted statistics table"""
    print("\n" + "="*80)
    print("HISTOGRAM ANALYSIS STATISTICS")
    print("="*80)
    
    for channel_name, channel_stats in stats.items():
        print(f"\n{channel_name} Channel:")
        print("-" * 60)
        print(f"{'Metric':<12} {'Original':<12} {'Encrypted':<12} {'Decrypted':<12}")
        print("-" * 60)
        
        metrics = ['mean', 'std', 'min', 'max', 'entropy']
        for metric in metrics:
            orig_val = channel_stats['Original'][metric]
            enc_val = channel_stats['Encrypted'][metric]
            dec_val = channel_stats['Decrypted'][metric]
            
            print(f"{metric.capitalize():<12} {orig_val:<12.2f} {enc_val:<12.2f} {dec_val:<12.2f}")

def evaluate_encryption_quality(original_image, encrypted_image, decrypted_image):
    """
    Evaluate encryption quality based on histogram analysis
    """
    print("\n" + "="*80)
    print("ENCRYPTION QUALITY EVALUATION")
    print("="*80)
    
    # Calculate correlation between original and encrypted
    orig_flat = original_image.flatten()
    enc_flat = encrypted_image.flatten()
    dec_flat = decrypted_image.flatten()
    
    # Correlation analysis
    orig_enc_corr = np.corrcoef(orig_flat, enc_flat)[0, 1]
    orig_dec_corr = np.corrcoef(orig_flat, dec_flat)[0, 1]
    
    print(f"Original-Encrypted Correlation: {orig_enc_corr:.6f}")
    print(f"Original-Decrypted Correlation: {orig_dec_corr:.6f}")
    
    # Perfect decryption check
    mse = np.mean((original_image.astype(float) - decrypted_image.astype(float))**2)
    psnr = 20 * np.log10(255 / np.sqrt(mse)) if mse > 0 else float('inf')
    
    print(f"MSE (Original vs Decrypted): {mse:.6f}")
    print(f"PSNR (Original vs Decrypted): {psnr:.2f} dB")
    
    # Histogram uniformity for encrypted image
    encrypted_hist = np.histogram(enc_flat, bins=256, range=(0, 256))[0]
    chi_square = np.sum((encrypted_hist - np.mean(encrypted_hist))**2) / np.mean(encrypted_hist)
    
    print(f"Chi-square test (Encrypted uniformity): {chi_square:.2f}")
    
    # Quality assessment
    print("\nQuality Assessment:")
    if abs(orig_enc_corr) < 0.1:
        print("✓ Good encryption: Low correlation between original and encrypted")
    else:
        print("⚠ Warning: High correlation between original and encrypted")
    
    if abs(orig_dec_corr) > 0.99:
        print("✓ Perfect decryption: High correlation between original and decrypted")
    else:
        print("⚠ Warning: Imperfect decryption")
    
    if mse < 1.0:
        print("✓ Excellent decryption quality: Very low MSE")
    else:
        print("⚠ Warning: High decryption error")

def run_complete_analysis(original_image, encrypted_image, decrypted_image, save_path=None):
    """
    Run complete histogram analysis including visualization and statistics
    """
    print("Starting comprehensive histogram analysis...")
    
    # Generate histogram plots
    fig = comprehensive_histogram_analysis(original_image, encrypted_image, decrypted_image, save_path)
    
    # Calculate and print statistics
    stats = calculate_histogram_statistics(original_image, encrypted_image, decrypted_image)
    print_histogram_statistics(stats)
    
    # Evaluate encryption quality
    evaluate_encryption_quality(original_image, encrypted_image, decrypted_image)
    
    return fig, stats

# Define output path for saving the histogram image (optional)
histogram_output_path = "histogram_output_tcca.png"

# Run complete analysis
fig, stats = run_complete_analysis(original_image, encrypted_image, decrypted_image, save_path=histogram_output_path)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import os

def calculate_correlation_coefficient(u, v):
    """
    Calculate correlation coefficient using the formula from your theory:
    ruv = cov(u,v) / sqrt(Du * Dv)
    
    Where:
    - cov(u,v) = E[(u-E(u))(v-E(v))] is the covariance of u and v
    - E(u) = (1/N) * sum(ui) is the expected value of u
    - Du = (1/N) * sum((ui-E(u))^2) is the variance of u
    """
    if len(u) != len(v):
        raise ValueError("Arrays must have the same length")
    
    N = len(u)
    
    # Calculate expected values E(u) and E(v)
    E_u = (1/N) * np.sum(u)
    E_v = (1/N) * np.sum(v)
    
    # Calculate variances Du and Dv
    Du = (1/N) * np.sum((u - E_u)**2)
    Dv = (1/N) * np.sum((v - E_v)**2)
    
    # Calculate covariance cov(u,v)
    cov_uv = (1/N) * np.sum((u - E_u) * (v - E_v))
    
    # Handle case where variance is zero (avoid division by zero)
    if Du == 0 or Dv == 0:
        return 0.0
    
    # Calculate correlation coefficient
    ruv = cov_uv / np.sqrt(Du * Dv)
    
    return ruv

def extract_adjacent_pixels(image_channel, direction='horizontal'):
    """
    Extract adjacent pixel pairs from an image channel in specified direction
    
    Parameters:
    image_channel: 2D numpy array representing single channel of image
    direction: 'horizontal', 'vertical', or 'diagonal'
    
    Returns:
    u, v: arrays of adjacent pixel values (following your notation)
    """
    h, w = image_channel.shape
    
    if direction == 'horizontal':
        # Adjacent horizontal pixels
        u = image_channel[:, :-1].flatten()  # All pixels except last column
        v = image_channel[:, 1:].flatten()   # All pixels except first column
    
    elif direction == 'vertical':
        # Adjacent vertical pixels
        u = image_channel[:-1, :].flatten()  # All pixels except last row
        v = image_channel[1:, :].flatten()   # All pixels except first row
    
    elif direction == 'diagonal':
        # Adjacent diagonal pixels
        u = image_channel[:-1, :-1].flatten()  # All pixels except last row and column
        v = image_channel[1:, 1:].flatten()    # All pixels except first row and column
    
    else:
        raise ValueError("Direction must be 'horizontal', 'vertical', or 'diagonal'")
    
    return u, v

def perform_correlation_analysis(original_image, encrypted_image, save_plots=True, output_dir='correlation_plots'):
    """
    Perform comprehensive correlation analysis between original and encrypted images
    using the correlation coefficient formula from your theory
    
    Parameters:
    original_image: numpy array of original RGB image
    encrypted_image: numpy array of encrypted RGB image
    save_plots: whether to save correlation plots
    output_dir: directory to save plots
    
    Returns:
    correlation_results: dictionary containing all correlation coefficients
    """
    
    # Create output directory if it doesn't exist
    if save_plots:
        os.makedirs(output_dir, exist_ok=True)
    
    # Ensure images have the same shape
    if original_image.shape != encrypted_image.shape:
        raise ValueError("Original and encrypted images must have the same dimensions")
    
    # Extract RGB channels
    channels = ['red', 'green', 'blue']
    directions = ['horizontal', 'vertical', 'diagonal']
    
    correlation_results = {}
    
    # Create a comprehensive plot with all correlation graphs
    fig, axes = plt.subplots(6, 3, figsize=(18, 24))
    fig.suptitle('Correlation Analysis of Images Before and After Encryption\n(Using Custom Correlation Formula)', fontsize=16, fontweight='bold')
    
    plot_labels = [
        ('a', 'b'), ('c', 'd'), ('e', 'f'),  # horizontal
        ('g', 'h'), ('i', 'j'), ('k', 'l'),  # vertical  
        ('m', 'n'), ('o', 'p'), ('q', 'r')   # diagonal
    ]
    
    plot_idx = 0
    
    print("Calculating correlation coefficients using custom formula:")
    print("ruv = cov(u,v) / sqrt(Du * Dv)")
    print("="*70)
    
    for dir_idx, direction in enumerate(directions):
        print(f"\n{direction.upper()} DIRECTION:")
        print("-" * 50)
        
        for ch_idx, channel in enumerate(channels):
            # Extract channel data
            orig_channel = original_image[:, :, ch_idx]
            enc_channel = encrypted_image[:, :, ch_idx]
            
            # Extract adjacent pixel pairs
            orig_u, orig_v = extract_adjacent_pixels(orig_channel, direction)
            enc_u, enc_v = extract_adjacent_pixels(enc_channel, direction)
            
            # Calculate correlation coefficients using custom formula
            orig_corr = calculate_correlation_coefficient(orig_u, orig_v)
            enc_corr = calculate_correlation_coefficient(enc_u, enc_v)
            
            # Store results
            key = f"{channel}_{direction}"
            correlation_results[key] = {
                'original_correlation': orig_corr,
                'encrypted_correlation': enc_corr,
                'original_pixels_u': orig_u,
                'original_pixels_v': orig_v,
                'encrypted_pixels_u': enc_u,
                'encrypted_pixels_v': enc_v,
                'sample_count': len(orig_u)  # N in the formula
            }
            
            # Create plots
            row = plot_idx // 3
            
            # Original image correlation plot
            ax_orig = axes[row * 2, ch_idx]
            ax_orig.scatter(orig_u, orig_v, alpha=0.1, s=0.5, c='blue')
            ax_orig.set_title(f'({plot_labels[plot_idx][0]}) {channel.capitalize()} {direction} (Original)\nCorr = {orig_corr:.6f}')
            ax_orig.set_xlabel('Pixel Value u(i)')
            ax_orig.set_ylabel('Pixel Value v(i)')
            ax_orig.grid(True, alpha=0.3)
            ax_orig.set_xlim(0, 255)
            ax_orig.set_ylim(0, 255)
            
            # Encrypted image correlation plot
            ax_enc = axes[row * 2 + 1, ch_idx]
            ax_enc.scatter(enc_u, enc_v, alpha=0.1, s=0.5, c='red')
            ax_enc.set_title(f'({plot_labels[plot_idx][1]}) {channel.capitalize()} {direction} (Encrypted)\nCorr = {enc_corr:.6f}')
            ax_enc.set_xlabel('Pixel Value u(i)')
            ax_enc.set_ylabel('Pixel Value v(i)')
            ax_enc.grid(True, alpha=0.3)
            ax_enc.set_xlim(0, 255)
            ax_enc.set_ylim(0, 255)
            
            print(f"{channel.capitalize()} channel:")
            print(f"  Sample count (N): {len(orig_u)}")
            print(f"  Original correlation: {orig_corr:.8f}")
            print(f"  Encrypted correlation: {enc_corr:.8f}")
            print(f"  Correlation reduction: {abs(orig_corr - enc_corr):.8f}")
        
        plot_idx += 3
    
    plt.tight_layout()
    
    if save_plots:
        plt.savefig(os.path.join(output_dir, 'correlation_analysis_custom_formula.png'), 
                   dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(output_dir, 'correlation_analysis_custom_formula.pdf'), 
                   bbox_inches='tight')
    
    plt.show()
    
    return correlation_results

def generate_correlation_table(correlation_results):
    """
    Generate Table 9 style correlation coefficient table as mentioned in your theory
    """
    print("\n" + "="*80)
    print("TABLE 9: CORRELATION COEFFICIENT VALUES")
    print("="*80)
    
    channels = ['red', 'green', 'blue']
    directions = ['horizontal', 'vertical', 'diagonal']
    
    # Create table header
    print(f"{'Direction':<12} {'Channel':<8} {'Original Image':<15} {'Encrypted Image':<15} {'Difference':<12}")
    print("-" * 80)
    
    total_orig_sum = 0
    total_enc_sum = 0
    count = 0
    
    for direction in directions:
        for i, channel in enumerate(channels):
            key = f"{channel}_{direction}"
            orig_corr = correlation_results[key]['original_correlation']
            enc_corr = correlation_results[key]['encrypted_correlation']
            diff = abs(orig_corr - enc_corr)
            
            # Display direction only for first channel
            dir_display = direction.capitalize() if i == 0 else ""
            
            print(f"{dir_display:<12} {channel.capitalize():<8} {orig_corr:<15.8f} {enc_corr:<15.8f} {diff:<12.8f}")
            
            total_orig_sum += abs(orig_corr)
            total_enc_sum += abs(enc_corr)
            count += 1
    
    print("-" * 80)
    avg_orig = total_orig_sum / count
    avg_enc = total_enc_sum / count
    print(f"{'Average':<21} {avg_orig:<15.8f} {avg_enc:<15.8f} {abs(avg_orig - avg_enc):<12.8f}")
    print()

def calculate_statistical_measures(u, v):
    """
    Calculate detailed statistical measures for correlation analysis
    Returns all intermediate values used in correlation calculation
    """
    N = len(u)
    
    # Expected values
    E_u = (1/N) * np.sum(u)
    E_v = (1/N) * np.sum(v)
    
    # Variances
    Du = (1/N) * np.sum((u - E_u)**2)
    Dv = (1/N) * np.sum((v - E_v)**2)
    
    # Covariance
    cov_uv = (1/N) * np.sum((u - E_u) * (v - E_v))
    
    # Correlation coefficient
    if Du == 0 or Dv == 0:
        ruv = 0.0
    else:
        ruv = cov_uv / np.sqrt(Du * Dv)
    
    return {
        'N': N,
        'E_u': E_u,
        'E_v': E_v,
        'Du': Du,
        'Dv': Dv,
        'cov_uv': cov_uv,
        'ruv': ruv
    }

def detailed_correlation_analysis(original_image, encrypted_image, channel_idx=0, direction='horizontal'):
    """
    Provide detailed step-by-step correlation calculation for one specific case
    This shows the actual implementation of your formulas
    """
    print("\n" + "="*80)
    print("DETAILED CORRELATION CALCULATION EXAMPLE")
    print("="*80)
    
    channel_names = ['Red', 'Green', 'Blue']
    print(f"Channel: {channel_names[channel_idx]}")
    print(f"Direction: {direction}")
    print()
    
    # Extract channel
    orig_channel = original_image[:, :, channel_idx]
    enc_channel = encrypted_image[:, :, channel_idx]
    
    # Extract adjacent pixels
    orig_u, orig_v = extract_adjacent_pixels(orig_channel, direction)
    enc_u, enc_v = extract_adjacent_pixels(enc_channel, direction)
    
    print("ORIGINAL IMAGE ANALYSIS:")
    print("-" * 40)
    orig_stats = calculate_statistical_measures(orig_u, orig_v)
    
    print(f"Sample size (N): {orig_stats['N']}")
    print(f"Expected value E(u): {orig_stats['E_u']:.6f}")
    print(f"Expected value E(v): {orig_stats['E_v']:.6f}")
    print(f"Variance Du: {orig_stats['Du']:.6f}")
    print(f"Variance Dv: {orig_stats['Dv']:.6f}")
    print(f"Covariance cov(u,v): {orig_stats['cov_uv']:.6f}")
    print(f"Correlation coefficient ruv: {orig_stats['ruv']:.8f}")
    
    print("\nENCRYPTED IMAGE ANALYSIS:")
    print("-" * 40)
    enc_stats = calculate_statistical_measures(enc_u, enc_v)
    
    print(f"Sample size (N): {enc_stats['N']}")
    print(f"Expected value E(u): {enc_stats['E_u']:.6f}")
    print(f"Expected value E(v): {enc_stats['E_v']:.6f}")
    print(f"Variance Du: {enc_stats['Du']:.6f}")
    print(f"Variance Dv: {enc_stats['Dv']:.6f}")
    print(f"Covariance cov(u,v): {enc_stats['cov_uv']:.6f}")
    print(f"Correlation coefficient ruv: {enc_stats['ruv']:.8f}")
    
    print("\nCORRELATION REDUCTION:")
    print("-" * 40)
    reduction = abs(orig_stats['ruv'] - enc_stats['ruv'])
    reduction_percent = (reduction / abs(orig_stats['ruv'])) * 100 if orig_stats['ruv'] != 0 else 0
    print(f"Absolute reduction: {reduction:.8f}")
    print(f"Percentage reduction: {reduction_percent:.2f}%")

def analyze_encryption_correlation(original_image, encrypted_image, output_dir='correlation_analysis'):
    """
    Complete correlation analysis workflow using your theoretical framework
    
    Parameters:
    original_image: numpy array of original image
    encrypted_image: numpy array of encrypted image  
    output_dir: directory to save results
    """
    
    print("CORRELATION ANALYSIS USING CUSTOM THEORETICAL FRAMEWORK")
    print("="*80)
    print("Formula: ruv = cov(u,v) / sqrt(Du * Dv)")
    print("Where:")
    print("  cov(u,v) = E[(u-E(u))(v-E(v))] - covariance")
    print("  E(u) = (1/N) * sum(ui) - expected value")
    print("  Du = (1/N) * sum((ui-E(u))^2) - variance")
    print("="*80)
    
    print(f"Original image shape: {original_image.shape}")
    print(f"Encrypted image shape: {encrypted_image.shape}")
    
    # Perform correlation analysis
    correlation_results = perform_correlation_analysis(
        original_image, encrypted_image, 
        save_plots=True, output_dir=output_dir
    )
    
    # Generate Table 9 style results
    generate_correlation_table(correlation_results)
    
    # Show detailed calculation for one example
    detailed_correlation_analysis(original_image, encrypted_image, 
                                channel_idx=0, direction='horizontal')
    
    # Security assessment
    print("\n" + "="*80)
    print("ENCRYPTION SECURITY ASSESSMENT")
    print("="*80)
    
    channels = ['red', 'green', 'blue']
    directions = ['horizontal', 'vertical', 'diagonal']
    
    all_encrypted_correlations = []
    for direction in directions:
        for channel in channels:
            key = f"{channel}_{direction}"
            enc_corr = abs(correlation_results[key]['encrypted_correlation'])
            all_encrypted_correlations.append(enc_corr)
    
    avg_enc_corr = np.mean(all_encrypted_correlations)
    max_enc_corr = np.max(all_encrypted_correlations)
    
    print(f"Average encrypted correlation: {avg_enc_corr:.8f}")
    print(f"Maximum encrypted correlation: {max_enc_corr:.8f}")
    
    if avg_enc_corr < 0.01:
        print("✓ EXCELLENT: Encryption effectively removes pixel correlation")
    elif avg_enc_corr < 0.05:
        print("✓ VERY GOOD: Very low correlation in encrypted image")
    elif avg_enc_corr < 0.1:
        print("✓ GOOD: Low correlation in encrypted image")
    else:
        print("⚠ WARNING: Significant correlation remains - consider improving encryption")
    
    # Save detailed results
    os.makedirs(output_dir, exist_ok=True)
    save_detailed_results(correlation_results, os.path.join(output_dir, 'detailed_correlation_results.txt'))
    
    print(f"\nAnalysis complete! Results saved to '{output_dir}' directory.")
    
    return correlation_results

def save_detailed_results(correlation_results, filename):
    """
    Save detailed correlation results to file
    """
    with open(filename, 'w') as f:
        f.write("CORRELATION COEFFICIENT ANALYSIS - DETAILED RESULTS\n")
        f.write("="*60 + "\n")
        f.write("Formula Used: ruv = cov(u,v) / sqrt(Du * Dv)\n")
        f.write("Where:\n")
        f.write("  cov(u,v) = E[(u-E(u))(v-E(v))]\n")
        f.write("  E(u) = (1/N) * sum(ui)\n")
        f.write("  Du = (1/N) * sum((ui-E(u))^2)\n\n")
        
        channels = ['red', 'green', 'blue']
        directions = ['horizontal', 'vertical', 'diagonal']
        
        for direction in directions:
            f.write(f"{direction.upper()} DIRECTION:\n")
            f.write("-" * 30 + "\n")
            for channel in channels:
                key = f"{channel}_{direction}"
                result = correlation_results[key]
                f.write(f"{channel.capitalize()} Channel:\n")
                f.write(f"  Sample Count (N): {result['sample_count']}\n")
                f.write(f"  Original Correlation: {result['original_correlation']:.10f}\n")
                f.write(f"  Encrypted Correlation: {result['encrypted_correlation']:.10f}\n")
                f.write(f"  Absolute Difference: {abs(result['original_correlation'] - result['encrypted_correlation']):.10f}\n\n")
            f.write("\n")

# Example usage - you would call this with your actual images
# def example_usage():
#     """
#     Example of how to use the correlation analysis with your images
#     Replace 'original_image' and 'encrypted_image' with your actual image arrays
#     """
#     # Load your images here
#     # original_image = np.array(Image.open('your_original_image.jpg'))
#     # encrypted_image = your_encryption_function(original_image, keys)
    
#     # For demonstration, create sample images
#     # You should replace this with your actual images
#     original_image = np.random.randint(0, 256, (256, 256, 3), dtype=np.uint8)
#     encrypted_image = np.random.randint(0, 256, (256, 256, 3), dtype=np.uint8)
    
#     # Run correlation analysis
#     correlation_results = analyze_encryption_correlation(original_image, encrypted_image)
    
#     return correlation_results

# Uncomment the following line to run the example
# results = example_usage()    
# Run correlation analysis between original and encrypted image
correlation_results =analyze_encryption_correlation(original_image, encrypted_image)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import os
from scipy import stats
import random

def calculate_npcr(C1, C2):
    """
    Calculate NPCR (Number of Pixels Change Rate) using equation (6):
    NPCR = (1/(R×C)) * Σ(r=1 to R)Σ(c=1 to C) Δ(r,c) × 100
    
    Where Δ(r,c) = 1 if C1(r,c) ≠ C2(r,c), else 0
    
    Parameters:
    C1, C2: Two encrypted images (numpy arrays)
    
    Returns:
    npcr_value: NPCR percentage
    """
    if C1.shape != C2.shape:
        raise ValueError("Images must have the same dimensions")
    
    # Handle both grayscale and color images
    if len(C1.shape) == 3:
        # For color images, check if any channel is different
        delta = np.any(C1 != C2, axis=2).astype(np.float64)
        R, C = C1.shape[:2]
    else:
        # For grayscale images
        delta = (C1 != C2).astype(np.float64)
        R, C = C1.shape
    
    # Calculate NPCR using equation (6)
    npcr = (1 / (R * C)) * np.sum(delta) * 100
    
    return npcr

def calculate_uaci(C1, C2):
    """
    Calculate UACI (Unified Average Changing Intensity) using equation (7):
    UACI = (1/(C×R)) * Σ(r=1 to C)Σ(c=1 to R) |C1(r,c) - C2(r,c)| / 255 × 100
    
    Parameters:
    C1, C2: Two encrypted images (numpy arrays)
    
    Returns:
    uaci_value: UACI percentage
    """
    if C1.shape != C2.shape:
        raise ValueError("Images must have the same dimensions")
    
    # Convert to float for calculations
    C1_float = C1.astype(np.float64)
    C2_float = C2.astype(np.float64)
    
    # Handle both grayscale and color images
    if len(C1.shape) == 3:
        # For color images, calculate for each channel and average
        R, C, channels = C1.shape
        total_diff = 0
        
        for ch in range(channels):
            diff = np.abs(C1_float[:, :, ch] - C2_float[:, :, ch])
            total_diff += np.sum(diff)
        
        # Average over all channels
        uaci = (1 / (R * C * channels)) * total_diff / 255 * 100
    else:
        # For grayscale images
        R, C = C1.shape
        diff = np.abs(C1_float - C2_float)
        uaci = (1 / (R * C)) * np.sum(diff) / 255 * 100
    
    return uaci

def create_single_pixel_change(image, position=None):
    """
    Create a copy of the image with a single pixel changed
    
    Parameters:
    image: Original image (numpy array)
    position: Tuple (row, col) for pixel position. If None, random position is chosen
    
    Returns:
    modified_image: Image with single pixel changed
    change_position: Position of the changed pixel
    original_value: Original pixel value
    new_value: New pixel value
    """
    modified_image = image.copy()
    
    if position is None:
        # Choose random position
        if len(image.shape) == 3:
            row = random.randint(0, image.shape[0] - 1)
            col = random.randint(0, image.shape[1] - 1)
        else:
            row = random.randint(0, image.shape[0] - 1)
            col = random.randint(0, image.shape[1] - 1)
        position = (row, col)
    
    row, col = position
    
    # Store original value
    if len(image.shape) == 3:
        original_value = image[row, col, :].copy()
        # Change pixel value (ensure it's different)
        new_value = original_value.copy()
        for ch in range(image.shape[2]):
            rand_int = random.randint(1, 255)
            new_val = (int(original_value[ch]) + rand_int) % 256
            new_value[ch] = np.uint8(new_val)

        modified_image[row, col, :] = new_value
    else:
        original_value = image[row, col]
        # Change pixel value (ensure it's different)
        new_value = (original_value + random.randint(1, 255)) % 256
        modified_image[row, col] = new_value
    
    return modified_image, position, original_value, new_value

def npcr_statistical_test(npcr_value, image_size, alpha=0.05):
    """
    Perform statistical test for NPCR following the methodology in equation (8)
    
    H0: N(C1,C2) = μN (null hypothesis - encryption is not secure)
    H1: N(C1,C2) < μN (alternative hypothesis - encryption is secure)
    
    Parameters:
    npcr_value: Calculated NPCR value
    image_size: Tuple (height, width) or (height, width, channels)
    alpha: Significance level (default 0.05)
    
    Returns:
    test_result: Dictionary with test results
    """
    if len(image_size) == 3:
        R, C, channels = image_size
        total_pixels = R * C
    else:
        R, C = image_size
        total_pixels = R * C
    
    # Theoretical expected NPCR for ideal encryption (99.6094% for large images)
    mu_N = 99.6094
    
    # Standard deviation for NPCR (theoretical)
    sigma_N = np.sqrt((mu_N * (100 - mu_N)) / total_pixels)
    
    # Calculate test statistic
    z_score = (npcr_value - mu_N) / sigma_N
    
    # Critical value for one-tailed test at significance level alpha
    z_critical = stats.norm.ppf(1 - alpha)
    
    # P-value for one-tailed test
    p_value = 1 - stats.norm.cdf(z_score)
    
    # Test decision
    reject_h0 = z_score > z_critical
    
    return {
        'npcr_value': npcr_value,
        'expected_npcr': mu_N,
        'standard_deviation': sigma_N,
        'z_score': z_score,
        'z_critical': z_critical,
        'p_value': p_value,
        'reject_h0': reject_h0,
        'conclusion': 'Secure against differential attacks' if reject_h0 else 'May be vulnerable to differential attacks',
        'alpha': alpha
    }

def uaci_statistical_test(uaci_value, image_size, alpha=0.05):
    """
    Perform statistical test for UACI
    
    Parameters:
    uaci_value: Calculated UACI value
    image_size: Tuple (height, width) or (height, width, channels)
    alpha: Significance level (default 0.05)
    
    Returns:
    test_result: Dictionary with test results
    """
    if len(image_size) == 3:
        R, C, channels = image_size
        total_pixels = R * C
    else:
        R, C = image_size
        total_pixels = R * C
    
    # Theoretical expected UACI for ideal encryption (33.4635% for large images)
    mu_U = 33.4635
    
    # Standard deviation for UACI (theoretical)
    sigma_U = np.sqrt((mu_U * (100 - mu_U)) / total_pixels)
    
    # Calculate test statistic
    z_score = (uaci_value - mu_U) / sigma_U
    
    # Critical values for two-tailed test at significance level alpha
    z_critical_lower = stats.norm.ppf(alpha / 2)
    z_critical_upper = stats.norm.ppf(1 - alpha / 2)
    
    # P-value for two-tailed test
    p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))
    
    # Test decision (for two-tailed test)
    reject_h0 = (z_score < z_critical_lower) or (z_score > z_critical_upper)
    
    return {
        'uaci_value': uaci_value,
        'expected_uaci': mu_U,
        'standard_deviation': sigma_U,
        'z_score': z_score,
        'z_critical_range': (z_critical_lower, z_critical_upper),
        'p_value': p_value,
        'reject_h0': reject_h0,
        'conclusion': 'Secure against differential attacks' if not reject_h0 else 'May be vulnerable to differential attacks',
        'alpha': alpha
    }

def comprehensive_differential_attack_test(encryption_function, original_image, num_tests=10, alpha=0.05):
    """
    Perform comprehensive differential attack analysis
    
    Parameters:
    encryption_function: Function that takes an image and returns encrypted image
    original_image: Original image (numpy array)
    num_tests: Number of single-pixel change tests to perform
    alpha: Significance level for statistical tests
    
    Returns:
    results: Dictionary containing all test results
    """
    print("COMPREHENSIVE DIFFERENTIAL ATTACK ANALYSIS")
    print("=" * 60)
    print(f"Image size: {original_image.shape}")
    print(f"Number of tests: {num_tests}")
    print(f"Significance level (α): {alpha}")
    print()
    
    npcr_values = []
    uaci_values = []
    test_details = []
    
    # Encrypt original image
    C1 = encryption_function(original_image)
    
    print("Performing single-pixel change tests...")
    print("-" * 40)
    
    for i in range(num_tests):
        print(f"Test {i+1}/{num_tests}: ", end="")
        
        # Create image with single pixel change
        modified_image, position, orig_val, new_val = create_single_pixel_change(original_image)
        
        # Encrypt modified image
        C2 = encryption_function(modified_image)
        
        # Calculate NPCR and UACI
        npcr = calculate_npcr(C1, C2)
        uaci = calculate_uaci(C1, C2)
        
        npcr_values.append(npcr)
        uaci_values.append(uaci)
        
        test_details.append({
            'test_number': i + 1,
            'change_position': position,
            'original_value': orig_val,
            'new_value': new_val,
            'npcr': npcr,
            'uaci': uaci
        })
        
        print(f"NPCR={npcr:.4f}%, UACI={uaci:.4f}%")
    
    # Calculate statistics
    npcr_mean = np.mean(npcr_values)
    npcr_std = np.std(npcr_values)
    uaci_mean = np.mean(uaci_values)
    uaci_std = np.std(uaci_values)
    
    # Perform statistical tests
    npcr_test = npcr_statistical_test(npcr_mean, original_image.shape, alpha)
    uaci_test = uaci_statistical_test(uaci_mean, original_image.shape, alpha)
    
    # Generate results
    results = {
        'test_parameters': {
            'image_shape': original_image.shape,
            'num_tests': num_tests,
            'alpha': alpha
        },
        'npcr_results': {
            'values': npcr_values,
            'mean': npcr_mean,
            'std': npcr_std,
            'min': np.min(npcr_values),
            'max': np.max(npcr_values),
            'statistical_test': npcr_test
        },
        'uaci_results': {
            'values': uaci_values,
            'mean': uaci_mean,
            'std': uaci_std,
            'min': np.min(uaci_values),
            'max': np.max(uaci_values),
            'statistical_test': uaci_test
        },
        'test_details': test_details,
        'overall_security': {
            'npcr_secure': npcr_test['reject_h0'],
            'uaci_secure': not uaci_test['reject_h0'],
            'overall_secure': npcr_test['reject_h0'] and not uaci_test['reject_h0']
        }
    }
    
    return results

def generate_differential_attack_report(results):
    """
    Generate comprehensive report for differential attack analysis
    """
    print("\n" + "=" * 80)
    print("DIFFERENTIAL ATTACK ANALYSIS REPORT")
    print("=" * 80)
    
    # Test parameters
    print("TEST PARAMETERS:")
    print("-" * 20)
    print(f"Image dimensions: {results['test_parameters']['image_shape']}")
    print(f"Number of tests: {results['test_parameters']['num_tests']}")
    print(f"Significance level (α): {results['test_parameters']['alpha']}")
    print()
    
    # NPCR Results
    print("NPCR (Number of Pixels Change Rate) ANALYSIS:")
    print("-" * 50)
    npcr = results['npcr_results']
    print(f"Mean NPCR: {npcr['mean']:.6f}%")
    print(f"Standard Deviation: {npcr['std']:.6f}%")
    print(f"Range: [{npcr['min']:.6f}%, {npcr['max']:.6f}%]")
    print(f"Theoretical ideal: {npcr['statistical_test']['expected_npcr']:.6f}%")
    print()
    
    print("NPCR Statistical Test Results:")
    npcr_test = npcr['statistical_test']
    print(f"  H0: NPCR = {npcr_test['expected_npcr']:.4f}% (not secure)")
    print(f"  H1: NPCR ≠ {npcr_test['expected_npcr']:.4f}% (secure)")
    print(f"  Z-score: {npcr_test['z_score']:.6f}")
    print(f"  Z-critical: {npcr_test['z_critical']:.6f}")
    print(f"  P-value: {npcr_test['p_value']:.6f}")
    print(f"  Decision: {'Reject H0' if npcr_test['reject_h0'] else 'Fail to reject H0'}")
    print(f"  Conclusion: {npcr_test['conclusion']}")
    print()
    
    # UACI Results
    print("UACI (Unified Average Changing Intensity) ANALYSIS:")
    print("-" * 55)
    uaci = results['uaci_results']
    print(f"Mean UACI: {uaci['mean']:.6f}%")
    print(f"Standard Deviation: {uaci['std']:.6f}%")
    print(f"Range: [{uaci['min']:.6f}%, {uaci['max']:.6f}%]")
    print(f"Theoretical ideal: {uaci['statistical_test']['expected_uaci']:.6f}%")
    print()
    
    print("UACI Statistical Test Results:")
    uaci_test = uaci['statistical_test']
    print(f"  H0: UACI = {uaci_test['expected_uaci']:.4f}% (secure)")
    print(f"  H1: UACI ≠ {uaci_test['expected_uaci']:.4f}% (not secure)")
    print(f"  Z-score: {uaci_test['z_score']:.6f}")
    print(f"  Z-critical range: [{uaci_test['z_critical_range'][0]:.6f}, {uaci_test['z_critical_range'][1]:.6f}]")
    print(f"  P-value: {uaci_test['p_value']:.6f}")
    print(f"  Decision: {'Reject H0' if uaci_test['reject_h0'] else 'Fail to reject H0'}")
    print(f"  Conclusion: {uaci_test['conclusion']}")
    print()
    
    # Overall Security Assessment
    print("OVERALL SECURITY ASSESSMENT:")
    print("-" * 35)
    overall = results['overall_security']
    print(f"NPCR Test: {'✓ PASS' if overall['npcr_secure'] else '✗ FAIL'}")
    print(f"UACI Test: {'✓ PASS' if overall['uaci_secure'] else '✗ FAIL'}")
    print(f"Overall Security: {'✓ SECURE' if overall['overall_secure'] else '✗ VULNERABLE'}")
    
    if overall['overall_secure']:
        print("\n🔒 The encryption algorithm is SECURE against differential attacks.")
        print("   Both NPCR and UACI values are within acceptable ranges.")
    else:
        print("\n⚠️  The encryption algorithm may be VULNERABLE to differential attacks.")
        print("   Consider improving the encryption algorithm.")

def create_differential_plots(results, save_path=None):
    """
    Create visualization plots for differential attack analysis
    """
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Differential Attack Analysis - NPCR and UACI Results', fontsize=16, fontweight='bold')
    
    npcr_values = results['npcr_results']['values']
    uaci_values = results['uaci_results']['values']
    
    # NPCR histogram
    ax1.hist(npcr_values, bins=min(10, len(npcr_values)), alpha=0.7, color='blue', edgecolor='black')
    ax1.axvline(results['npcr_results']['statistical_test']['expected_npcr'], 
                color='red', linestyle='--', linewidth=2, label='Theoretical Ideal')
    ax1.axvline(results['npcr_results']['mean'], 
                color='green', linestyle='-', linewidth=2, label='Measured Mean')
    ax1.set_xlabel('NPCR (%)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('NPCR Distribution')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # UACI histogram
    ax2.hist(uaci_values, bins=min(10, len(uaci_values)), alpha=0.7, color='orange', edgecolor='black')
    ax2.axvline(results['uaci_results']['statistical_test']['expected_uaci'], 
                color='red', linestyle='--', linewidth=2, label='Theoretical Ideal')
    ax2.axvline(results['uaci_results']['mean'], 
                color='green', linestyle='-', linewidth=2, label='Measured Mean')
    ax2.set_xlabel('UACI (%)')
    ax2.set_ylabel('Frequency')
    ax2.set_title('UACI Distribution')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # NPCR vs Test Number
    test_numbers = range(1, len(npcr_values) + 1)
    ax3.plot(test_numbers, npcr_values, 'bo-', alpha=0.7, label='NPCR Values')
    ax3.axhline(results['npcr_results']['statistical_test']['expected_npcr'], 
                color='red', linestyle='--', linewidth=2, label='Theoretical Ideal')
    ax3.axhline(results['npcr_results']['mean'], 
                color='green', linestyle='-', linewidth=2, label='Mean')
    ax3.set_xlabel('Test Number')
    ax3.set_ylabel('NPCR (%)')
    ax3.set_title('NPCR Values Across Tests')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # UACI vs Test Number
    ax4.plot(test_numbers, uaci_values, 'ro-', alpha=0.7, label='UACI Values')
    ax4.axhline(results['uaci_results']['statistical_test']['expected_uaci'], 
                color='red', linestyle='--', linewidth=2, label='Theoretical Ideal')
    ax4.axhline(results['uaci_results']['mean'], 
                color='green', linestyle='-', linewidth=2, label='Mean')
    ax4.set_xlabel('Test Number')
    ax4.set_ylabel('UACI (%)')
    ax4.set_title('UACI Values Across Tests')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.savefig(save_path.replace('.png', '.pdf'), bbox_inches='tight')
    
    plt.show()

def save_differential_results(results, filename='differential_attack_results.txt'):
    """
    Save detailed differential attack results to file
    """
    with open(filename, 'w') as f:
        f.write("DIFFERENTIAL ATTACK ANALYSIS - DETAILED RESULTS\n")
        f.write("=" * 60 + "\n\n")
        
        # Test parameters
        f.write("TEST PARAMETERS:\n")
        f.write(f"Image dimensions: {results['test_parameters']['image_shape']}\n")
        f.write(f"Number of tests: {results['test_parameters']['num_tests']}\n")
        f.write(f"Significance level: {results['test_parameters']['alpha']}\n\n")
        
        # Individual test results
        f.write("INDIVIDUAL TEST RESULTS:\n")
        f.write("-" * 30 + "\n")
        f.write(f"{'Test':<6} {'Position':<15} {'NPCR (%)':<12} {'UACI (%)':<12}\n")
        f.write("-" * 50 + "\n")
        
        for detail in results['test_details']:
            pos_str = f"({detail['change_position'][0]},{detail['change_position'][1]})"
            f.write(f"{detail['test_number']:<6} {pos_str:<15} {detail['npcr']:<12.6f} {detail['uaci']:<12.6f}\n")
        
        f.write("\n")
        
        # Statistical summary
        f.write("STATISTICAL SUMMARY:\n")
        f.write("-" * 20 + "\n")
        f.write(f"NPCR Mean: {results['npcr_results']['mean']:.8f}%\n")
        f.write(f"NPCR Std: {results['npcr_results']['std']:.8f}%\n")
        f.write(f"UACI Mean: {results['uaci_results']['mean']:.8f}%\n")
        f.write(f"UACI Std: {results['uaci_results']['std']:.8f}%\n\n")
        
        # Security assessment
        f.write("SECURITY ASSESSMENT:\n")
        f.write("-" * 20 + "\n")
        f.write(f"NPCR Test: {'PASS' if results['overall_security']['npcr_secure'] else 'FAIL'}\n")
        f.write(f"UACI Test: {'PASS' if results['overall_security']['uaci_secure'] else 'FAIL'}\n")
        f.write(f"Overall: {'SECURE' if results['overall_security']['overall_secure'] else 'VULNERABLE'}\n")

# Example usage function
# def example_differential_test():
#     """
#     Example of how to use the differential attack tests
#     Replace with your actual encryption function
#     """
    
#     # Example encryption function (replace with your actual function)
#     def sample_encryption(image):
#         # This is just a sample - replace with your actual encryption
#         np.random.seed(42)  # For reproducibility in example
#         return np.random.randint(0, 256, image.shape, dtype=np.uint8)
    
#     # Create sample image (replace with your actual image)
#     original_image = np.random.randint(0, 256, (128, 128, 3), dtype=np.uint8)
    
#     # Run comprehensive differential attack test
#     results = comprehensive_differential_attack_test(
#         encryption_function=sample_encryption,
#         original_image=original_image,
#         num_tests=20,
#         alpha=0.05
#     )
    
#     # Generate report
#     generate_differential_attack_report(results)
    
#     # Create plots
#     create_differential_plots(results, 'differential_attack_analysis.png')
    
#     # Save results
#     save_differential_results(results, 'differential_results.txt')
    
#     return results

# Uncomment to run example
# results = example_differential_test()

In [ ]:
# 3. Run comprehensive differential attack test
key = keys_seq  # Replace with actual key used in your encryption pipeline

results = comprehensive_differential_attack_test(
    encryption_function=lambda img: encrypt_image(img, key),
    original_image=original_image,
    num_tests=50,
    alpha=0.05
)

# 4. Generate reports and plots
generate_differential_attack_report(results)
create_differential_plots(results, 'npcr_uaci_analysis.png')
save_differential_results(results, 'differential_results.txt')

In [ ]:
test_image_path = '/home/gunjan/Encryption/baboon.jpg'
# test_image_path = '/home/gunjan/Encryption/lena.bmp'
# test_image_path = '/home/gunjan/Encryption/pepper.jpg'
# test_image_path = '/home/gunjan/Encryption/barbara.jpg'
# test_image_path = '/home/gunjan/Encryption/pepper.jpg'
# test_image_path = '/home/gunjan/Encryption/Lena_256.png'
# test_image_path = '/home/gunjan/Encryption/dataset/chest-x-ray.jpg'
# test_image_path = '/home/gunjan/output_images/image_08572.png'

test_image = Image.open(test_image_path)
test_image = np.array(test_image)


test_image.shape

In [ ]:
# Start measuring time
start_time = time.time()

# Call the encryption function
encrypted_test_image = encrypt_image(test_image, keys_seq)

# End measuring time
end_time = time.time()

# Calculate and print the encryption time
encryption_time = end_time - start_time
print(f"Total Encryption Time: {encryption_time:.6f} seconds")

In [ ]:
display_image(test_image, "test")
display_image(encrypted_test_image, "encrypted")
display_image(decrypted_test_image, "decrypted")

In [ ]:


# Parameters
mu_range = np.arange(-10, 10.01, 0.01)  # Range of control parameter values
num_iterations = 150
num_transient = 100  # Transient iterations to discard

# Arrays to store bifurcation diagram data
bifurcation_diag = []
parameter_values = []

# Bifurcation diagram generation
for mu in mu_range:
    x = 0.25454  # Initial condition
    # Transient iterations to stabilize the system
    for _ in range(num_transient):
        x = 1 - 2 * (np.cos(np.arccos(x) * np.exp(abs(mu)) * np.arccos(x)))**2
    
    # Generate points for the bifurcation diagram
    for _ in range(num_iterations):
        x = 1 - 2 * (np.cos(np.arccos(x) * np.exp(abs(mu)) * np.arccos(x)))**2
        bifurcation_diag.append([mu, x])
        parameter_values.append(mu)

# Convert bifurcation diagram data to a NumPy array for easy slicing
bifurcation_diag = np.array(bifurcation_diag)

# Plotting the bifurcation diagram
plt.figure(figsize=(8, 6))
plt.tight_layout()
plt.plot(bifurcation_diag[:, 0], bifurcation_diag[:, 1], '.', markersize=0.4)
plt.title('Bifurcation Diagram for 1-DEC Chaotic Map')
plt.xlabel(r'$\mu$ (Control Parameter)')
plt.ylabel(r'$x_{n+1}$')
plt.ylim(ymin=-1, ymax=1)
plt.xlim(xmin=-10, xmax=10)
plt.yticks(np.linspace(-1, 1, 11))
# plt.grid()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Function to generate bifurcation data for a given map
def generate_bifurcation_data(map_func, mu_range, num_transient=100, num_iterations=150, x_init=0.25454):
    bifurcation_diag = []
    for mu in mu_range:
        x = x_init
        # Transient iterations to remove transient effects
        for _ in range(num_transient):
            x = map_func(x, mu)
        
        # Record stable iterations
        for _ in range(num_iterations):
            x = map_func(x, mu)
            bifurcation_diag.append([mu, x])
    return np.array(bifurcation_diag)

# 1-DEC chaotic map
def dec_map(x, mu):
    return 1 - 2 * (np.cos(np.arccos(x) * np.exp(abs(mu)) * np.arccos(x)))**2

# 1D Chebyshev map
def chebyshev_map(x, mu):
    return np.cos(mu * np.arccos(x))

# Logistic map
def logistic_map(x, mu):
    return mu * x * (1 - x)

# 1DCP map
def one_dcp_map(x, mu):
    return np.cos(mu * (x**3 + x))

# Range of mu values
mu_range_dec = np.arange(-10, 10.01, 0.01)
mu_range_chebyshev = np.arange(0.1, 2.0, 0.01)
mu_range_logistic = np.arange(2.5, 4.0, 0.01)

# Generate bifurcation data
bifurcation_dec = generate_bifurcation_data(dec_map, mu_range_dec, x_init=0.25454)
bifurcation_chebyshev = generate_bifurcation_data(chebyshev_map, mu_range_chebyshev, x_init=0.5)
bifurcation_logistic = generate_bifurcation_data(logistic_map, mu_range_logistic, x_init=0.5)

# Plot all three bifurcation diagrams
fig, axes = plt.subplots(2, 1, figsize=(5,8))

# # Plot 1-DEC chaotic map
# axes[0].plot(bifurcation_dec[:, 0], bifurcation_dec[:, 1], '.', markersize=0.2)
# axes[0].set_title('Bifurcation Diagram for 1-DEC Chaotic Map')
# axes[0].set_xlabel(r'$\mu$ (Control Parameter)')
# axes[0].set_ylabel(r'$x_{n+1}$')

# Plot 1D Chebyshev map
axes[0].plot(bifurcation_chebyshev[:, 0], bifurcation_chebyshev[:, 1], '.', markersize=0.2)
axes[0].set_title('Bifurcation Diagram for 1D Chebyshev Map')
axes[0].set_xlabel(r'$\mu$ (Control Parameter)')
axes[0].set_ylabel(r'$x_{n+1}$')

# Plot Logistic map
axes[1].plot(bifurcation_logistic[:, 0], bifurcation_logistic[:, 1], '.', markersize=0.2)
axes[1].set_title('Bifurcation Diagram for Logistic Map')
axes[1].set_xlabel(r'$\mu$ (Control Parameter)')
axes[1].set_ylabel(r'$x_{n+1}$')

plt.tight_layout()
plt.show()


In [ ]:
# Define Lyapunov Exponent Calculation Function
def calculate_lyapunov_exponent(map_function, param_range, x0, iterations, transient):
    lyapunov_exponents = []
    params = np.linspace(param_range[0], param_range[1], 1000)
    
    for mu in params:
        x = x0
        le_sum = 0
        for _ in range(transient):  # Discard transient iterations
            x = map_function(x, mu)
        
        for _ in range(iterations):  # Lyapunov exponent calculation iterations
            try:
                dx = abs(map_function(x + 1e-8, mu) - map_function(x, mu)) / 1e-8
                le_sum += np.log(abs(dx))
                x = map_function(x, mu)
            except ZeroDivisionError:
                le_sum += -np.inf

        lyapunov_exponents.append(le_sum / iterations)
    
    return params, lyapunov_exponents

# 1-DEC chaotic map
def dec_map(x, mu):
    return 1 - 2 * (np.cos(np.arccos(x) * np.exp(abs(mu)) * np.arccos(x)))**2

def chebyshev_map(x, mu):
    return np.cos(mu * np.arccos(x))

def logistic_map(x, mu):
    return mu * x * (1 - x)
    
def one_dcp_map(x, mu):
    return np.cos(mu * (x**3 + x))

# Define the new chaotic map with parameters alpha and beta
def new_chaotic_map(x, alpha, beta):
    return (x * (alpha + 1)) ** (np.sin(beta * np.pi + x))

# Compute Lyapunov Exponent for the new chaotic map
alpha_values = np.linspace(0.01, 5, 500) # np.linspace(0.01, 2, 200)  # Control parameter alpha
beta = 0.5  # Fixed beta for simplicity

lyapunov_exponents_new_map = []

for alpha in alpha_values:
    x = 0.5  # Initial condition
    lyapunov_sum = 0
    num_iterations = 1000
    transient = 200  # Discard initial transient

    for i in range(num_iterations):
        derivative = (alpha + 1) ** (np.sin(beta * np.pi + x)) * (
            np.log(x * (alpha + 1)) * np.cos(beta * np.pi + x) * np.pi
        )
        if i >= transient:
            lyapunov_sum += np.log(abs(derivative) + 1e-10)  # Avoid log(0)
        x = new_chaotic_map(x, alpha, beta)

    lyapunov_exponents_new_map.append(lyapunov_sum / (num_iterations - transient))
    
# Parameters
param_range = (0, 5)
x0 = 0.01
iterations = 500
transient = 100
dsp_range = np.arange(-5, 5.01, 0.01)

# Calculate Lyapunov exponents for the maps
params_dec, le_dec = calculate_lyapunov_exponent(dec_map, param_range, x0, iterations, transient)
params_chebyshev, le_chebyshev = calculate_lyapunov_exponent(chebyshev_map, param_range, x0, iterations, transient)
params_logistic, le_logistic = calculate_lyapunov_exponent(logistic_map, (0, 5), x0, iterations, transient)
params_1dcp, le_1dcp = calculate_lyapunov_exponent(one_dcp_map, param_range, x0, iterations, transient)
# Plotting
plt.figure(figsize=(8,6))
plt.plot(params_dec, le_dec, label="1-DEC Map", color="blue")
plt.plot(params_chebyshev, le_chebyshev, label="Chebyshev Map", color="green")
plt.plot(params_logistic, le_logistic, label="Logistic Map", color="red")
plt.plot(params_1dcp, le_1dcp, label="1-DCP Map")
plt.plot(alpha_values, lyapunov_exponents_new_map, label="1-DSP (α, β=0.5)", color="orange")
plt.axhline(0, color="black", linestyle="--", linewidth=0.2)
# plt.title("Lyapunov Exponents of Chaotic Maps")
plt.xlabel("Control Parameter (\u03BC)")
plt.ylabel("Lyapunov Exponent")
# plt.ylim(ymin=-1, ymax=1)
plt.xlim(xmin=param_range[0], xmax=param_range[1])
plt.legend()
# plt.grid()
plt.show()

In [ ]:
# Function to calculate NPCR and UACI
def calculate_npcr_uaci(image1, image2):
    # Ensure images are of type double for calculations
    image1 = image1.astype(np.float64)
    image2 = image2.astype(np.float64)
    
    # Check if the image is grayscale or RGB
    if len(image1.shape) == 2:  # Grayscale
        diff_pixels = np.sum(image1 != image2)
        total_pixels = image1.shape[0] * image1.shape[1]
        npcr = (diff_pixels / total_pixels) * 100
        uaci = (np.sum(np.abs(image1 - image2)) / (total_pixels * 255)) * 100
        return npcr, uaci
    else:  # RGB
        npcr_results = []
        uaci_results = []
        for channel in range(image1.shape[2]):  # Loop over channels
            channel1 = image1[:, :, channel]
            channel2 = image2[:, :, channel]
            diff_pixels = np.sum(channel1 != channel2)
            total_pixels = channel1.shape[0] * channel1.shape[1]
            npcr = (diff_pixels / total_pixels) * 100
            uaci = (np.sum(np.abs(channel1 - channel2)) / (total_pixels * 255)) * 100
            npcr_results.append(npcr)
            uaci_results.append(uaci)
        return np.mean(npcr_results), np.mean(uaci_results)

# Function to calculate NPCR and UACI for R, G, B components separately
def calculate_rgb_npcr_uaci(image1, image2):
    # Ensure images are of type double for calculations
    image1 = image1.astype(np.float64)
    image2 = image2.astype(np.float64)
    
    # Check if the image is grayscale or RGB
    if len(image1.shape) == 2:  # Grayscale
        diff_pixels = np.sum(image1 != image2)
        total_pixels = image1.shape[0] * image1.shape[1]
        npcr = (diff_pixels / total_pixels) * 100
        uaci = (np.sum(np.abs(image1 - image2)) / (total_pixels * 255)) * 100
        return npcr, uaci
    else:  # RGB
        npcr_r, npcr_g, npcr_b = [], [], []  # Lists to store NPCR for R, G, B
        uaci_r, uaci_g, uaci_b = [], [], []  # Lists to store UACI for R, G, B
        
        for channel in range(image1.shape[2]):  # Loop over channels (R, G, B)
            channel1 = image1[:, :, channel]
            channel2 = image2[:, :, channel]
            
            # Calculate NPCR for the current channel
            diff_pixels = np.sum(channel1 != channel2)
            total_pixels = channel1.shape[0] * channel1.shape[1]
            npcr = (diff_pixels / total_pixels) * 100
            uaci = (np.sum(np.abs(channel1 - channel2)) / (total_pixels * 255)) * 100
            
            # Append results for each channel
            if channel == 0:  # Red channel
                npcr_r.append(npcr)
                uaci_r.append(uaci)
            elif channel == 1:  # Green channel
                npcr_g.append(npcr)
                uaci_g.append(uaci)
            else:  # Blue channel
                npcr_b.append(npcr)
                uaci_b.append(uaci)
        
        # Return separate NPCR and UACI values for each channel
        return npcr_r, npcr_g, npcr_b, uaci_r, uaci_g, uaci_b


### DeepFool Adversarial Attack NPCR AND UACI Test

In [ ]:
orig_image_path = '/home/gunjan/Encryption/adversarial_images/original_0_label_5.png'
adv_image_path = '/home/gunjan/Encryption/adversarial_images/adversarial_0_label_5.png'

In [ ]:

orig_image = Image.open(orig_image_path)
# orig_image = orig_image.resize((128, 128))
orig_image = np.array(orig_image)


orig_image.shape

# Call the encryption function
encrypted_orig_image = encrypt_image(orig_image, keys_seq)

In [ ]:
decrypted_orig_image = decrypt_image(encrypted_orig_image, keys_seq)


In [ ]:
display_image(orig_image, "test")
display_image(encrypted_orig_image, "encrypted")
display_image(decrypted_orig_image, "decrypted")

In [ ]:
adv_image = Image.open(adv_image_path)
# adv_image = adv_image.resize((128, 128))
adv_image = np.array(adv_image)


adv_image.shape

# Call the encryption function
encrypted_adv_image = encrypt_image(adv_image, keys_seq)
decrypted_adv_image = decrypt_image(encrypted_adv_image, keys_seq)

display_image(adv_image, "test")
display_image(encrypted_adv_image, "encrypted")
display_image(decrypted_adv_image, "decrypted")

In [ ]:
print(calculate_npcr_uaci(encrypted_orig_image, encrypted_adv_image))

In [ ]:
# Convert the array to a PIL Image
encrypted_adv_pil= Image.fromarray(encrypted_adv_image)
encrypted_orig_pil= Image.fromarray(encrypted_orig_image)

adv_pil= Image.fromarray(adv_image)
orig_pil= Image.fromarray(orig_image)

# Save the image to a file
encrypted_adv_pil_path = "encrypted_adv_pil.png"
encrypted_orig_pil_path = "encrypted_orig_pil.png"

encrypted_adv_pil.save(encrypted_adv_pil_path)
encrypted_orig_pil.save(encrypted_orig_pil_path)

print(f"Image saved successfully to {encrypted_adv_pil_path} and {encrypted_orig_pil_path}")

In [ ]:
adv_pil= Image.fromarray(adv_image)
orig_pil= Image.fromarray(orig_image)

# Save the image to a file
adv_pil_path = "adv_pil.png"
orig_pil_path = "orig_pil.png"

adv_pil.save(adv_pil_path)
orig_pil.save(orig_pil_path)

In [ ]:
decrypted_adv_pil= Image.fromarray(decrypted_adv_image)
decrypted_orig_pil= Image.fromarray(decrypted_orig_image)

# Save the image to a file
decrypted_adv_pil_path = "decrypted_adv_pil.png"
decrypted_orig_pil_path = "decrypted_orig_pil.png"

decrypted_adv_pil.save(decrypted_adv_pil_path)
decrypted_orig_pil.save(decrypted_orig_pil_path)

In [ ]:
print("DeepFool attack NPCR AND UACI: ", calculate_rgb_npcr_uaci(encrypted_orig_image, encrypted_adv_image)) #DeepFool

### FGSM Adversarial Attack NPCR AND UACI Test

In [ ]:
orig_image_path = '/home/gunjan/Encryption/adversarial_images/original_image.png'
adv_image_path = '/home/gunjan/Encryption/adversarial_images/adversarial_image.png'

In [ ]:
orig_image = Image.open(orig_image_path)
# orig_image = orig_image.resize((128, 128))
orig_image = np.array(orig_image)


orig_image.shape

# Call the encryption function
encrypted_orig_image = encrypt_image(orig_image, keys_seq)

In [ ]:
adv_image = Image.open(adv_image_path)
# adv_image = adv_image.resize((128, 128))
adv_image = np.array(adv_image)


adv_image.shape

# Call the encryption function
encrypted_adv_image = encrypt_image(adv_image, keys_seq)

In [ ]:
display_image(orig_image, "test")
display_image(encrypted_orig_image, "encrypted")
display_image(adv_image, "test")
display_image(encrypted_adv_image, "encrypted")

In [ ]:
print("FGSM attack NPCR AND UACI: ", calculate_rgb_npcr_uaci(encrypted_orig_image, encrypted_adv_image)) #FGSM